In [1]:
#year 8 final data set
import pyreadstat
import pandas as pd

def filter_active_lives(sav_path, la_col, survey_year):
    """
    Filters Active Lives Survey SPSS file to London boroughs
    with all four analytical pillars.
    
    Parameters:
        sav_path    : full path to the .sav file
        la_col      : LA column to use for this year e.g. 'LA_2023'
        survey_year : label for this dataset e.g. '2022-23'
    
    Returns:
        df_london   : filtered dataframe
        report      : dictionary with summary info
    """

    print(f"\n{'='*60}")
    print(f"Processing Year: {survey_year}")
    print(f"{'='*60}")

    # ── Step 1: Load metadata only ────────────────────────────────
    print("Step 1: Loading metadata...")
    _, meta = pyreadstat.read_sav(sav_path, metadataonly=True)
    label_dict   = meta.column_names_to_labels
    file_vars    = set(meta.column_names)
    value_labels = meta.variable_value_labels

    print(f"  Total columns in file: {len(file_vars)}")

    # ── Step 2: Define all pillar columns ─────────────────────────
    print("Step 2: Defining pillar columns...")

    # CORE
    core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
            'MonthTwelve', 'group', 'mode']

    # GEOGRAPHY — all LA columns plus the one for this year
    geo = [v for v in file_vars if v.startswith('LA_')]

    # ── P1: Participation ─────────────────────────────────────────
    p1_prefixes = [
        'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
        'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
        'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
        'DURATION', 'DURATIONGR', 'MINS_SESS',
        'ACTYRA', 'ACTYRB', 'ACTYRC',
        'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
    ]
    p1_exact_names = [
        'MEMS7_ALL', 'MEMS7GR_ALL',
        'MEMS7_SPORTCOUNT_A01', 'MEMS7GR_SPORTCOUNT_A01',
        'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
        'Number_Activities_150', 'Number_Activities_Gr2',
        'Number_Activities_Gr5', 'Number_Activities_150_Gr2',
        'Number_Activities_150_Gr5', 'DAYS10P60GR',
    ]
    p1_exact     = [v for v in file_vars if v in p1_exact_names or v in p1_prefixes]
    p1_composite = [v for v in file_vars if
                    any(v.startswith(p) for p in p1_prefixes) and
                    any(s in v for s in ['_C0', '_B0', '_D0']) and
                    'GARD' not in v.upper()]  # exclude gardening
    club_summary = [v for v in file_vars if
                    v.startswith('CLUB_') and
                    any(v.endswith(s) for s in [
                        '_A01','_A02','_A03',
                        '_B01','_B03','_B05','_B06','_B07',
                        '_C01','_C02','_C04','_C05','_C06',
                        '_C07','_C08','_C09','_C10','_C11',
                        '_C13','_C14','_C15'
                    ])]
    club_summary += [v for v in [
        'CLUB_SPORTCOUNT_A01', 'Number_Club', 'Number_Club_Gr2',
        'Club_ExcFitness', 'CLUB_SPORTFUND_A02'
    ] if v in file_vars]
    barriers     = [v for v in file_vars if v.startswith('limfreti')]
    p1           = list(set(p1_exact + p1_composite + club_summary + barriers))

    # ── P2: Demographics ──────────────────────────────────────────
    demo = [v for v in [
        'Age9', 'Age5_2',
        'Gend3', 'GendAge9',
        'Eth7', 'Eth2', 'EthAge4',
        'Disab3', 'Disab2_POP',
        'NSSEC5', 'NSSEC8', 'Educ6',
    ] if v in file_vars]

    health = [v for v in [
        'health', 'BMIG',
        'disty1_POP', 'disty2_POP', 'disty3_POP', 'disty4_POP',
        'disty5_POP', 'disty6_POP', 'disty7_POP', 'disty8_POP',
        'disty9_POP', 'disty10_POP',
    ] if v in file_vars]

    attitudes = [v for v in [
        'READYAB1_POP', 'READYOP1_POP',
        'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
    ] if v in file_vars]

    p2 = list(set(demo + health + attitudes))

    # ── P3: Indoor / Outdoor ──────────────────────────────────────
    mems_inout = [v for v in file_vars if
                  any(v.startswith(p) for p in
                      ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
                  'SPORTCOUNT_A01' in v]
    inclus     = [v for v in file_vars if
                  v.lower().startswith('inclus') and 'GR2' not in v]
    p3         = list(set(mems_inout + inclus))

    # ── P4: Volunteering ──────────────────────────────────────────
    p4 = [v for v in [
        'VolAny',
        'volint1_vol', 'volint2_vol', 'volint3_vol', 'volint4_vol',
        'volint5_vol', 'volint6_vol', 'volint7_vol',
        'VolFrqB_Pop', 'VolDur_GR2', 'VolLong_GR3',
        'VolCnt', 'VolCnt_GR2',
    ] if v in file_vars]

    # ── Step 3: Combine and validate ──────────────────────────────
    print("Step 3: Combining and validating columns...")
    all_cols     = list(set(core + geo + p1 + p2 + p3 + p4))
    valid_cols   = [v for v in all_cols if v in file_vars]
    missing_cols = [v for v in all_cols if v not in file_vars]

    print(f"  Core:          {len([v for v in core if v in file_vars])}")
    print(f"  Geography:     {len([v for v in geo if v in file_vars])}")
    print(f"  P1:            {len(p1)}")
    print(f"  P2:            {len(p2)}")
    print(f"  P3:            {len(p3)}")
    print(f"  P4:            {len(p4)}")
    print(f"  Total unique:  {len(valid_cols)}")
    if missing_cols:
        print(f"  Missing cols:  {missing_cols}")

    # ── Step 4: Load data ─────────────────────────────────────────
    print("Step 4: Loading data (may take a minute)...")
    df, _ = pyreadstat.read_sav(
        sav_path,
        usecols=valid_cols,
        apply_value_formats=False
    )
    print(f"  Full dataset shape: {df.shape}")

    # ── Step 5: Filter to London (excl. City of London) ───────────
    print("Step 5: Filtering to London boroughs...")
    borough_labels  = value_labels.get(la_col, {})
    london_boroughs = {code: name for code, name in borough_labels.items()
                       if 'E09' in str(name) and 'E09000001' not in str(name)}
    london_codes    = list(london_boroughs.keys())

    if not london_codes:
        print(f"  WARNING: No London codes found in {la_col} — check LA column!")
        return None, None

    df_london = df[df[la_col].isin(london_codes)].copy()

    # Add survey year label
    df_london['survey_year'] = survey_year

    # Standardise LA column name
    df_london = df_london.rename(columns={la_col: 'LA_code'})

    print(f"  London boroughs found: {len(london_boroughs)}")
    print(f"  London rows:           {df_london.shape[0]:,}")
    print(f"  Final shape:           {df_london.shape}")

    # ── Step 6: Build report ──────────────────────────────────────
    report = {
        'survey_year':      survey_year,
        'total_rows':       df_london.shape[0],
        'total_cols':       df_london.shape[1],
        'p1_cols':          len(p1),
        'p2_cols':          len(p2),
        'p3_cols':          len(p3),
        'p4_cols':          len(p4),
        'missing_cols':     missing_cols,
        'london_boroughs':  len(london_boroughs),
    }

    return df_london, report


# ── Usage — Year 8 (2022-23) ──────────────────────────────────────────────────
df_y8, report_y8 = filter_active_lives(
    sav_path    = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
    la_col      = 'LA_2023',
    survey_year = '2022-23'
)

# Save Year 8
df_y8.to_csv('active_lives_london_y8_2022-23.csv', index=False)
print(f"\n✓ Saved: active_lives_london_y8_2022-23.csv")
print(f"  Report: {report_y8}")


Processing Year: 2022-23
Step 1: Loading metadata...
  Total columns in file: 10737
Step 2: Defining pillar columns...
Step 3: Combining and validating columns...
  Core:          8
  Geography:     6
  P1:            521
  P2:            30
  P3:            22
  P4:            13
  Total unique:  600
Step 4: Loading data (may take a minute)...
  Full dataset shape: (172968, 600)
Step 5: Filtering to London boroughs...
  London boroughs found: 32
  London rows:           16,515
  Final shape:           (16515, 601)

✓ Saved: active_lives_london_y8_2022-23.csv
  Report: {'survey_year': '2022-23', 'total_rows': 16515, 'total_cols': 601, 'p1_cols': 521, 'p2_cols': 30, 'p3_cols': 22, 'p4_cols': 13, 'missing_cols': [], 'london_boroughs': 32}


In [3]:
# col check with year 6&7
import pyreadstat
import pandas as pd

# ── Define diagnostic function ────────────────────────────────────────────────
def compare_year_columns(sav_path, survey_year, reference_cols):
    print(f"\n{'='*60}")
    print(f"Checking: {survey_year}")
    print(f"{'='*60}")

    _, meta = pyreadstat.read_sav(sav_path, metadataonly=True)
    file_vars = set(meta.column_names)

    reference_set = set(reference_cols)

    missing = reference_set - file_vars
    new     = file_vars - reference_set
    common  = reference_set & file_vars

    print(f"  Total cols in file:     {len(file_vars)}")
    print(f"  Year 8 reference cols:  {len(reference_set)}")
    print(f"  Common cols (matched):  {len(common)}")
    print(f"  Missing from this year: {len(missing)}")
    print(f"  New cols not in Year 8: {len(new)}")

    if missing:
        print(f"\n  ⚠ MISSING COLS (in Y8 but not here):")
        for v in sorted(missing):
            print(f"    - {v}")

    return {
        'survey_year': survey_year,
        'total_cols':  len(file_vars),
        'common':      len(common),
        'missing':     sorted(missing),
        'new':         len(new),
    }

# ── Year 8 reference columns ──────────────────────────────────────────────────
reference_cols = pd.read_csv('active_lives_london_y8_2022-23.csv').columns.tolist()
reference_cols.remove('survey_year')
reference_cols.remove('LA_code')
print(f"Year 8 reference: {len(reference_cols)} columns")

# ── Run diagnostic on Years 6 and 7 ──────────────────────────────────────────
year_files = {
    '2021-22': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav", 'LA_2021'),
    '2020-21': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav", 'LA_2020'),
}

reports = {}
for year, (path, la_col) in year_files.items():
    reports[year] = compare_year_columns(path, year, reference_cols)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
for year, r in reports.items():
    print(f"\n{year}:")
    print(f"  Total cols:   {r['total_cols']}")
    print(f"  Common:       {r['common']}")
    print(f"  Missing:      {len(r['missing'])}")
    if r['missing']:
        print(f"  Missing list: {r['missing']}")

Year 8 reference: 599 columns

Checking: 2021-22
  Total cols in file:     10488
  Year 8 reference cols:  599
  Common cols (matched):  588
  Missing from this year: 11
  New cols not in Year 8: 9900

  ⚠ MISSING COLS (in Y8 but not here):
    - inclus_a
    - inclus_b
    - inclus_c
    - limfreti1
    - limfreti2
    - limfreti3
    - limfreti4
    - limfreti5
    - limfreti6
    - limfreti7
    - limfreti8

Checking: 2020-21
  Total cols in file:     10505
  Year 8 reference cols:  599
  Common cols (matched):  587
  Missing from this year: 12
  New cols not in Year 8: 9918

  ⚠ MISSING COLS (in Y8 but not here):
    - READYOP1_POP
    - inclus_a
    - inclus_b
    - inclus_c
    - limfreti1
    - limfreti2
    - limfreti3
    - limfreti4
    - limfreti5
    - limfreti6
    - limfreti7
    - limfreti8

SUMMARY

2021-22:
  Total cols:   10488
  Common:       588
  Missing:      11
  Missing list: ['inclus_a', 'inclus_b', 'inclus_c', 'limfreti1', 'limfreti2', 'limfreti3', 'limfreti4'

In [4]:
# Year 7 (2021-22) fianl data
df_y7, report_y7 = filter_active_lives(
    sav_path    = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav",
    la_col      = 'LA_2021',
    survey_year = '2021-22'
)
df_y7.to_csv('active_lives_london_y7_2021-22.csv', index=False)
print(f"✓ Saved Y7: {df_y7.shape}")

# Year 6 (2020-21) final data
df_y6, report_y6 = filter_active_lives(
    sav_path    = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav",
    la_col      = 'LA_2020',
    survey_year = '2020-21'
)
df_y6.to_csv('active_lives_london_y6_2020-21.csv', index=False)
print(f"✓ Saved Y6: {df_y6.shape}")


Processing Year: 2021-22
Step 1: Loading metadata...
  Total columns in file: 10488
Step 2: Defining pillar columns...
Step 3: Combining and validating columns...
  Core:          8
  Geography:     6
  P1:            513
  P2:            30
  P3:            19
  P4:            13
  Total unique:  589
Step 4: Loading data (may take a minute)...
  Full dataset shape: (177551, 589)
Step 5: Filtering to London boroughs...
  London boroughs found: 32
  London rows:           16,139
  Final shape:           (16139, 590)
✓ Saved Y7: (16139, 590)

Processing Year: 2020-21
Step 1: Loading metadata...
  Total columns in file: 10505
Step 2: Defining pillar columns...
Step 3: Combining and validating columns...
  Core:          8
  Geography:     6
  P1:            513
  P2:            29
  P3:            19
  P4:            13
  Total unique:  588
Step 4: Loading data (may take a minute)...
  Full dataset shape: (177273, 588)
Step 5: Filtering to London boroughs...
  London boroughs found: 32
 

In [5]:
# ── Diagnostic for Years 1-5 ──────────────────────────────────────────────────
year_files_remaining = {
    '2019-20': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav", 'LA_2019'),
    '2018-19': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav", 'LA_2019'),
    '2017-18': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav", 'LA_2015'),
    '2016-17': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav", 'LA_2015'),
    '2015-16': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav", 'LA_2015'),
}

reports = {}
for year, (path, la_col) in year_files_remaining.items():
    reports[year] = compare_year_columns(path, year, reference_cols)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SUMMARY — Years 1-5")
print(f"{'='*60}")
for year, r in reports.items():
    print(f"\n{year}:")
    print(f"  Total cols:  {r['total_cols']}")
    print(f"  Common:      {r['common']}")
    print(f"  Missing:     {len(r['missing'])}")
    if r['missing']:
        print(f"  Missing list: {r['missing']}")


Checking: 2019-20
  Total cols in file:     10428
  Year 8 reference cols:  599
  Common cols (matched):  586
  Missing from this year: 13
  New cols not in Year 8: 9842

  ⚠ MISSING COLS (in Y8 but not here):
    - CLUB_INFORMAL_C15
    - READYOP1_POP
    - inclus_a
    - inclus_b
    - inclus_c
    - limfreti1
    - limfreti2
    - limfreti3
    - limfreti4
    - limfreti5
    - limfreti6
    - limfreti7
    - limfreti8

Checking: 2018-19
  Total cols in file:     9114
  Year 8 reference cols:  599
  Common cols (matched):  535
  Missing from this year: 64
  New cols not in Year 8: 8579

  ⚠ MISSING COLS (in Y8 but not here):
    - CLUB_INFORMAL_C15
    - DAYS10PGR4x_CYCALL_C02
    - DAYS10PGR4x_EXBIKE_CYCLECLASS_D04
    - DAYS10PGR4x_GroupEx_D04
    - DAYS10P_GroupEx_D04
    - MUSCLE7GR_ACTTRAV_C03
    - MUSCLE7GR_ADVENTURE_D01
    - MUSCLE7GR_ADVWATERSPORT_C07
    - MUSCLE7GR_ATHLETICS_D03
    - MUSCLE7GR_COMBATTARGET_C09
    - MUSCLE7GR_CYCALL_C02
    - MUSCLE7GR_CYCLEISSPORT_B03

In [7]:
import pyreadstat
import pandas as pd

# ── Load Year 8 metadata (reference) ─────────────────────────────────────────
SAV_Y8 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav"
_, meta_y8 = pyreadstat.read_sav(SAV_Y8, metadataonly=True)
label_dict_y8 = meta_y8.column_names_to_labels

# ── Load Year 5 (2019-20) metadata ───────────────────────────────────────────
SAV_Y5 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav"
_, meta_y5 = pyreadstat.read_sav(SAV_Y5, metadataonly=True)
label_dict_y5 = meta_y5.column_names_to_labels
file_vars_y5  = set(meta_y5.column_names)

# ── Search for each missing column ───────────────────────────────────────────
missing_y5 = [
    'CLUB_INFORMAL_C15', 'READYOP1_POP',
    'inclus_a', 'inclus_b', 'inclus_c',
    'limfreti1', 'limfreti2', 'limfreti3', 'limfreti4',
    'limfreti5', 'limfreti6', 'limfreti7', 'limfreti8'
]

print("Searching for alternatives in 2019-20 data...\n")
for col in missing_y5:
    keyword = col.split('_')[0][:8]
    matches = [(v, label_dict_y5.get(v, '')) for v in file_vars_y5
               if keyword.upper() in v.upper() or
               keyword.lower() in label_dict_y5.get(v, '').lower()]

    print(f"\n--- Missing: {col} ---")
    print(f"    Y8 label: {label_dict_y8.get(col, 'no label')}")
    if matches:
        print(f"    Possible matches in Y5:")
        for v, l in sorted(matches)[:5]:
            print(f"      {v:40} | {l}")
    else:
        print(f"    ✗ Genuinely absent from Y5")

Searching for alternatives in 2019-20 data...


--- Missing: CLUB_INFORMAL_C15 ---
    Y8 label: Whether member of a club: Informal Activity and Active Play
    Possible matches in Y5:
      CLUB_ADVENTURE_D01                       | Whether member of a club: Adventure sports
      CLUB_ADVWATERSPORT_C07                   | Whether member of a club: Adventure, outdoor and water sports
      CLUB_AIKIDO_S04                          | Whether member of a club: Aikido
      CLUB_AIRGUN_S08                          | Whether member of a club: Airgun (including pistol)
      CLUB_ANGLING_I01                         | Whether member of a club: Angling

--- Missing: READYOP1_POP ---
    Y8 label: Readiness for activity: Opportunity (REBASED)
    ✗ Genuinely absent from Y5

--- Missing: inclus_a ---
    Y8 label: I find the places and environments where I exercise inclusive and welcoming
    ✗ Genuinely absent from Y5

--- Missing: inclus_b ---
    Y8 label: I see people who are similar to me 

In [9]:
# year 5 final data
import pyreadstat
import pandas as pd

SAV_Y5 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav"

# ── Step 1: Load metadata ─────────────────────────────────────────────────────
_, meta_y5 = pyreadstat.read_sav(SAV_Y5, metadataonly=True)
file_vars_y5 = set(meta_y5.column_names)

# ── Step 2: Define columns (same logic as filter_active_lives) ────────────────
core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
        'MonthTwelve', 'group', 'mode']
geo  = [v for v in file_vars_y5 if v.startswith('LA_')]

p1_prefixes = [
    'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
    'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
    'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
    'DURATION', 'DURATIONGR', 'MINS_SESS',
    'ACTYRA', 'ACTYRB', 'ACTYRC',
    'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
]
p1_exact_names = [
    'MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_SPORTCOUNT_A01',
    'MEMS7GR_SPORTCOUNT_A01', 'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
    'Number_Activities_150', 'Number_Activities_Gr2', 'Number_Activities_Gr5',
    'Number_Activities_150_Gr2', 'Number_Activities_150_Gr5', 'DAYS10P60GR',
]
p1_exact     = [v for v in file_vars_y5 if v in p1_exact_names or v in p1_prefixes]
p1_composite = [v for v in file_vars_y5 if
                any(v.startswith(p) for p in p1_prefixes) and
                any(s in v for s in ['_C0', '_B0', '_D0']) and
                'GARD' not in v.upper()]
club_summary = [v for v in file_vars_y5 if
                v.startswith('CLUB_') and
                any(v.endswith(s) for s in [
                    '_A01','_A02','_A03','_B01','_B03','_B05','_B06','_B07',
                    '_C01','_C02','_C04','_C05','_C06','_C07','_C08','_C09',
                    '_C10','_C11','_C13','_C14','_C15'
                ])]
club_summary += [v for v in ['CLUB_SPORTCOUNT_A01','Number_Club','Number_Club_Gr2',
                              'Club_ExcFitness','CLUB_SPORTFUND_A02'] if v in file_vars_y5]
barriers     = [v for v in file_vars_y5 if v.startswith('limfreti')]
p1           = list(set(p1_exact + p1_composite + club_summary + barriers))

demo = [v for v in [
    'Age9', 'Age5_2', 'Gend3', 'GendAge9',
    'Eth7', 'Eth2', 'EthAge4',
    'Disab3', 'Disab2_POP', 'NSSEC5', 'NSSEC8', 'Educ6',
] if v in file_vars_y5]

health = [v for v in [
    'health', 'BMIG',
    'disty1_POP','disty2_POP','disty3_POP','disty4_POP',
    'disty5_POP','disty6_POP','disty7_POP','disty8_POP',
    'disty9_POP','disty10_POP',
] if v in file_vars_y5]

# ── Y5 specific: use renamed READYOP variable ─────────────────────────────────
attitudes = [v for v in [
    'READYAB1_POP',
    'READYOP_CV_3_POP',   # renamed version of READYOP1_POP in Y5
    'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
] if v in file_vars_y5]

p2 = list(set(demo + health + attitudes))

mems_inout = [v for v in file_vars_y5 if
              any(v.startswith(p) for p in ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
              'SPORTCOUNT_A01' in v]
inclus     = [v for v in file_vars_y5 if
              v.lower().startswith('inclus') and 'GR2' not in v]
p3         = list(set(mems_inout + inclus))

p4 = [v for v in [
    'VolAny',
    'volint1_vol','volint2_vol','volint3_vol','volint4_vol',
    'volint5_vol','volint6_vol','volint7_vol',
    'VolFrqB_Pop','VolDur_GR2','VolLong_GR3','VolCnt','VolCnt_GR2',
] if v in file_vars_y5]

# ── Step 3: Combine and validate ──────────────────────────────────────────────
all_cols   = list(set(core + geo + p1 + p2 + p3 + p4))
valid_cols = [v for v in all_cols if v in file_vars_y5]

print(f"Core:          {len([v for v in core if v in file_vars_y5])}")
print(f"Geography:     {len(geo)}")
print(f"P1:            {len(p1)}")
print(f"P2:            {len(p2)}")
print(f"P3:            {len(p3)}")
print(f"P4:            {len(p4)}")
print(f"Total unique:  {len(valid_cols)}")

# ── Step 4: Load data ─────────────────────────────────────────────────────────
print("\nLoading data...")
df_y5, _ = pyreadstat.read_sav(SAV_Y5, usecols=valid_cols, apply_value_formats=False)
print(f"Full dataset shape: {df_y5.shape}")

# ── Step 5: Rename Y5-specific column to standard name ───────────────────────
df_y5 = df_y5.rename(columns={'READYOP_CV_3_POP': 'READYOP1_POP'})
print("Renamed READYOP_CV_3_POP → READYOP1_POP ✓")

# ── Step 6: Filter to London (excl. City of London) ──────────────────────────
borough_labels  = meta_y5.variable_value_labels.get('LA_2019', {})
london_boroughs = {code: name for code, name in borough_labels.items()
                   if 'E09' in str(name) and 'E09000001' not in str(name)}
london_codes    = list(london_boroughs.keys())

df_y5_london = df_y5[df_y5['LA_2019'].isin(london_codes)].copy()

# ── Step 7: Standardise and save ─────────────────────────────────────────────
df_y5_london['survey_year'] = '2019-20'
df_y5_london = df_y5_london.rename(columns={'LA_2019': 'LA_code'})

print(f"\nLondon boroughs: {len(london_boroughs)}")
print(f"London rows:     {df_y5_london.shape[0]:,}")
print(f"Final shape:     {df_y5_london.shape}")

df_y5_london.to_csv('active_lives_london_y5_2019-20.csv', index=False)
print("\n✓ Saved: active_lives_london_y5_2019-20.csv")

Core:          8
Geography:     6
P1:            512
P2:            30
P3:            19
P4:            13
Total unique:  588

Loading data...
Full dataset shape: (177735, 588)
Renamed READYOP_CV_3_POP → READYOP1_POP ✓

London boroughs: 32
London rows:     16,091
Final shape:     (16091, 589)

✓ Saved: active_lives_london_y5_2019-20.csv


In [11]:
import pandas as pd

y6 = pd.read_csv('active_lives_london_y6_2020-21.csv')
y7 = pd.read_csv('active_lives_london_y7_2021-22.csv')
y8 = pd.read_csv('active_lives_london_y8_2022-23.csv')

y8_cols = set(y8.columns)

def classify(col):
    if col.startswith('limfreti'):
        return 'P1 - Free Time Barriers'
    if any(col.startswith(p) for p in ['MUSCLE7','DAYS10P','FREQUENCYGR2','MINS_SESS',
                                        'CLUB_','Club_','Number_Activities','Number_Club',
                                        'ACTYRA','ACTYRB','ACTYRC']):
        return 'P1 - Composite'
    if any(col.startswith(p) for p in ['MEMS7GR_IN','MEMS7GR_OUT','inclus']):
        return 'P3 - Indoor/Outdoor'
    if col in ['READYAB1_POP','READYOP1_POP','Motiva_POP',
               'motivb_POP','motivc_POP','motivd_POP','health']:
        return 'P2 - Attitudes/Health'
    if col.startswith('Vol') or col.startswith('vol'):
        return 'P4 - Volunteering'
    if col.startswith('LA_') or col in ['MonthTwelve','month','survey_year']:
        return 'Core/Geo'
    return 'Other'

for label, df in [('Year 6 (2020-21)', y6), ('Year 7 (2021-22)', y7)]:
    missing = sorted(y8_cols - set(df.columns))
    by_pillar = {}
    for col in missing:
        pillar = classify(col)
        by_pillar.setdefault(pillar, []).append(col)

    print(f"\n{'='*60}")
    print(f"{label} — {df.shape[0]:,} rows, {df.shape[1]} cols")
    print(f"Missing vs Y8: {len(missing)} columns")
    print(f"{'='*60}")
    for pillar, cols in sorted(by_pillar.items()):
        print(f"\n  {pillar} ({len(cols)} missing):")
        for c in cols:
            print(f"    - {c}")


Year 6 (2020-21) — 16,028 rows, 589 cols
Missing vs Y8: 13 columns

  Core/Geo (1 missing):
    - LA_2020

  P1 - Free Time Barriers (8 missing):
    - limfreti1
    - limfreti2
    - limfreti3
    - limfreti4
    - limfreti5
    - limfreti6
    - limfreti7
    - limfreti8

  P2 - Attitudes/Health (1 missing):
    - READYOP1_POP

  P3 - Indoor/Outdoor (3 missing):
    - inclus_a
    - inclus_b
    - inclus_c

Year 7 (2021-22) — 16,139 rows, 590 cols
Missing vs Y8: 12 columns

  Core/Geo (1 missing):
    - LA_2021

  P1 - Free Time Barriers (8 missing):
    - limfreti1
    - limfreti2
    - limfreti3
    - limfreti4
    - limfreti5
    - limfreti6
    - limfreti7
    - limfreti8

  P3 - Indoor/Outdoor (3 missing):
    - inclus_a
    - inclus_b
    - inclus_c


In [13]:
#check for year 4
# ── Load Year 4 (2018-19) metadata ───────────────────────────────────────────
SAV_Y4 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav"
_, meta_y4 = pyreadstat.read_sav(SAV_Y4, metadataonly=True)
label_dict_y4 = meta_y4.column_names_to_labels
file_vars_y4  = set(meta_y4.column_names)

# Search for key missing variables by concept
searches = ['MUSCLE', 'volint', 'VolDur', 'VolFrq', 'VolLon',
            'INFORMAL', 'READYOP', 'inclus', 'limfre', 'strengthen',
            'free time', 'barrier']

print("Searching 2018-19 SPSS metadata...\n")
for keyword in searches:
    # Search both variable names and labels
    matches = [(v, label_dict_y4.get(v) or '') for v in file_vars_y4
           if keyword.upper() in v.upper() or
           keyword.lower() in (label_dict_y4.get(v) or '').lower()]
    print(f"\n{keyword} ({len(matches)} matches):")
    for v, l in sorted(matches)[:5]:
        print(f"  {v:45} | {l}")
    if not matches:
        print(f"  ✗ Nothing found")

Searching 2018-19 SPSS metadata...


MUSCLE (0 matches):
  ✗ Nothing found

volint (9 matches):
  VolInt1_JUST                                  | Just this role: Raise funds for a sports club, organisation or event
  VolInt_ANY                                    | Whether done any volunteering in last 12 months
  volint1                                       | During the last 12 months, have you given any of your time to do any of the following activities?: Raise funds for a sports club, organisation or event
  volint2                                       | During the last 12 months, have you given any of your time to do any of the following activities?: Provide transport which helps people take part in sport (other than family members)
  volint3                                       | During the last 12 months, have you given any of your time to do any of the following activities?: Coach or instruct an individual or team(s) in a sport or recreational physical activity (other than sol

In [14]:
cnt_vars = [(v, (label_dict_y4.get(v) or '')) for v in file_vars_y4
            if 'volc' in v.lower() or 'vol_c' in v.lower() or 
            'count' in (label_dict_y4.get(v) or '').lower()]

print("VolCnt related:")
for v, l in sorted(cnt_vars):
    print(f"  {v:35} | {l}")
    

VolCnt related:
  ACT7GR_SPORTCOUNT_A01               | Activity: MEMS (grouped, with Inactivity split): Sport (count definition)
  ACTYRA_SPORTCOUNT_A01               | Done 1-3 months ago: Sport (count definition)
  ACTYRB_SPORTCOUNT_A01               | Done 4-6 months ago: Sport (count definition)
  ACTYRC_SPORTCOUNT_A01               | Done 7-12 months ago: Sport (count definition)
  ACTYR_3_MOUNTAIN_H01                | Participation pattern throughout the year in 3 groups: Sport (count definition)
  ACTYR_3_SPORTCOUNT_A01              | Participation pattern throughout the year in 3 groups: Sport (count definition)
  ACTYR_4_MOUNTAIN_H01                | Participation pattern throughout the year in 4 groups: Sport (count definition)
  ACTYR_4_SPORTCOUNT_A01              | Participation pattern throughout the year in 4 groups: Sport (count definition)
  ACTYR_7_MOUNTAIN_H01                | Participation pattern throughout the year in 7 groups: Sport (count definition)
  ACTYR_7_S

In [15]:
import pyreadstat
import pandas as pd

SAV_Y4 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav"

# ── Step 1: Load metadata ─────────────────────────────────────────────────────
_, meta_y4 = pyreadstat.read_sav(SAV_Y4, metadataonly=True)
file_vars_y4 = set(meta_y4.column_names)

# ── Step 2: Define columns ────────────────────────────────────────────────────
core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
        'MonthTwelve', 'group', 'mode']
geo  = [v for v in file_vars_y4 if v.startswith('LA_')]

p1_prefixes = [
    'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
    'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
    'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
    'DURATION', 'DURATIONGR', 'MINS_SESS',
    'ACTYRA', 'ACTYRB', 'ACTYRC',
    'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
]
p1_exact_names = [
    'MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_SPORTCOUNT_A01',
    'MEMS7GR_SPORTCOUNT_A01', 'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
    'Number_Activities_150', 'Number_Activities_Gr2', 'Number_Activities_Gr5',
    'Number_Activities_150_Gr2', 'Number_Activities_150_Gr5', 'DAYS10P60GR',
]
p1_exact     = [v for v in file_vars_y4 if v in p1_exact_names or v in p1_prefixes]
p1_composite = [v for v in file_vars_y4 if
                any(v.startswith(p) for p in p1_prefixes) and
                any(s in v for s in ['_C0', '_B0', '_D0']) and
                'GARD' not in v.upper()]
club_summary = [v for v in file_vars_y4 if
                v.startswith('CLUB_') and
                any(v.endswith(s) for s in [
                    '_A01','_A02','_A03','_B01','_B03','_B05','_B06','_B07',
                    '_C01','_C02','_C04','_C05','_C06','_C07','_C08','_C09',
                    '_C10','_C11','_C13','_C14','_C15'
                ])]
club_summary += [v for v in ['CLUB_SPORTCOUNT_A01','Number_Club','Number_Club_Gr2',
                              'Club_ExcFitness','CLUB_SPORTFUND_A02'] if v in file_vars_y4]
barriers     = [v for v in file_vars_y4 if v.startswith('limfreti')]
p1           = list(set(p1_exact + p1_composite + club_summary + barriers))

demo = [v for v in [
    'Age9', 'Age5_2', 'Gend3', 'GendAge9',
    'Eth7', 'Eth2', 'EthAge4',
    'Disab3', 'Disab2_POP', 'NSSEC5', 'NSSEC8', 'Educ6',
] if v in file_vars_y4]

health = [v for v in [
    'health', 'BMIG',
    'disty1_POP','disty2_POP','disty3_POP','disty4_POP',
    'disty5_POP','disty6_POP','disty7_POP','disty8_POP',
    'disty9_POP','disty10_POP',
] if v in file_vars_y4]

attitudes = [v for v in [
    'READYAB1_POP', 'READYOP1_POP',
    'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
] if v in file_vars_y4]

p2 = list(set(demo + health + attitudes))

mems_inout = [v for v in file_vars_y4 if
              any(v.startswith(p) for p in ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
              'SPORTCOUNT_A01' in v]
inclus     = [v for v in file_vars_y4 if
              v.lower().startswith('inclus') and 'GR2' not in v]
p3         = list(set(mems_inout + inclus))

# ── Y4 specific: use renamed volunteering variables ───────────────────────────
p4 = [v for v in [
    'VolInt_ANY',       # will be renamed to VolAny
    'volint1',          # will be renamed to volint1_vol
    'volint2',
    'volint3',
    'volint4',
    'volint5',
    'volint6',
    'volint7',
    'VolFrq_POP',       # will be renamed to VolFrqB_Pop
    'VolCnt', 'VolCnt_GR2',
    # VolDur_GR2 and VolLong_GR3 genuinely absent
] if v in file_vars_y4]

# ── Step 3: Combine and validate ──────────────────────────────────────────────
all_cols   = list(set(core + geo + p1 + p2 + p3 + p4))
valid_cols = [v for v in all_cols if v in file_vars_y4]

print(f"Core:          {len([v for v in core if v in file_vars_y4])}")
print(f"Geography:     {len(geo)}")
print(f"P1:            {len(p1)}")
print(f"P2:            {len(p2)}")
print(f"P3:            {len(p3)}")
print(f"P4:            {len(p4)}")
print(f"Total unique:  {len(valid_cols)}")

# ── Step 4: Load data ─────────────────────────────────────────────────────────
print("\nLoading data...")
df_y4, _ = pyreadstat.read_sav(SAV_Y4, usecols=valid_cols, apply_value_formats=False)
print(f"Full dataset shape: {df_y4.shape}")

# ── Step 5: Rename Y4-specific columns to standard names ─────────────────────
rename_map = {
    'VolInt_ANY': 'VolAny',
    'volint1':    'volint1_vol',
    'volint2':    'volint2_vol',
    'volint3':    'volint3_vol',
    'volint4':    'volint4_vol',
    'volint5':    'volint5_vol',
    'volint6':    'volint6_vol',
    'volint7':    'volint7_vol',
    'VolFrq_POP': 'VolFrqB_Pop',
}
df_y4 = df_y4.rename(columns=rename_map)
print(f"Renamed {len(rename_map)} columns ✓")

# ── Step 6: Filter to London (excl. City of London) ──────────────────────────
borough_labels  = meta_y4.variable_value_labels.get('LA_2019', {})
london_boroughs = {code: name for code, name in borough_labels.items()
                   if 'E09' in str(name) and 'E09000001' not in str(name)}
london_codes    = list(london_boroughs.keys())

df_y4_london = df_y4[df_y4['LA_2019'].isin(london_codes)].copy()

# ── Step 7: Standardise and save ─────────────────────────────────────────────
df_y4_london['survey_year'] = '2018-19'
df_y4_london = df_y4_london.rename(columns={'LA_2019': 'LA_code'})

print(f"\nLondon boroughs: {len(london_boroughs)}")
print(f"London rows:     {df_y4_london.shape[0]:,}")
print(f"Final shape:     {df_y4_london.shape}")

df_y4_london.to_csv('active_lives_london_y4_2018-19.csv', index=False)
print("\n✓ Saved: active_lives_london_y4_2018-19.csv")


Core:          8
Geography:     6
P1:            470
P2:            30
P3:            19
P4:            11
Total unique:  544

Loading data...
Full dataset shape: (181535, 544)
Renamed 9 columns ✓

London boroughs: 32
London rows:     15,889
Final shape:     (15889, 545)

✓ Saved: active_lives_london_y4_2018-19.csv


In [16]:
import pyreadstat

# ── Load metadata for Years 1, 2, 3 ──────────────────────────────────────────
year_paths = {
    'Y3_2017-18': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav",
    'Y2_2016-17': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav",
    'Y1_2015-16': r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav",
}

# ── Variables to verify — all suspected renames ───────────────────────────────
# Format: {standard_name: [possible_names_to_search]}
to_check = {
    # Volunteering
    'VolAny':       ['VolAny', 'VOLANY', 'Volany', 'VolInt_ANY', 'Volint_ANY'],
    'VolFrqB_Pop':  ['VolFrqB_Pop', 'VolFrq_POP', 'VOLFRQ_POP', 'Volfrq_POP'],
    'VolCnt_GR2':   ['VolCnt_GR2', 'Volcnt_GR2', 'VOLCNT_GR2'],
    'VolCnt':       ['VolCnt', 'Volcnt', 'VOLCNT'],
    'volint1_vol':  ['volint1_vol', 'volint1', 'VolInt1_JUST', 'volint1_just'],
    'volint2_vol':  ['volint2_vol', 'volint2', 'VolInt2_JUST', 'volint2_just'],
    'volint3_vol':  ['volint3_vol', 'volint3', 'VolInt3_JUST', 'volint3_just'],
    'volint4_vol':  ['volint4_vol', 'volint4', 'VolInt4_JUST', 'volint4_just'],
    'volint5_vol':  ['volint5_vol', 'volint5', 'VolInt5_JUST', 'volint5_just'],
    'volint6_vol':  ['volint6_vol', 'volint6', 'VolInt6_JUST', 'volint6_just'],
    'volint7_vol':  ['volint7_vol', 'volint7', 'VolInt7_JUST', 'volint7_just'],
    # Core
    'MonthTwelve':  ['MonthTwelve', 'MONTHTWELVE', 'monthtwelve'],
    'health':       ['health', 'HEALTH', 'Health'],
    # P3
    'MEMS7GR_IN_SPORTCOUNT_A01':  ['MEMS7GR_IN_SPORTCOUNT_A01', 'MEMS7GR_INTSESSION_E05'],
    'MEMS7GR_OUT_SPORTCOUNT_A01': ['MEMS7GR_OUT_SPORTCOUNT_A01'],
}

# ── Run check for each year ───────────────────────────────────────────────────
for year_label, path in year_paths.items():
    print(f"\n{'='*60}")
    print(f"{year_label}")
    print(f"{'='*60}")

    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    file_vars    = set(meta.column_names)
    label_dict   = meta.column_names_to_labels

    for standard_name, candidates in to_check.items():
        found = [(c, label_dict.get(c, '')) for c in candidates if c in file_vars]
        if not found:
            print(f"  ✗ {standard_name:<25} — NOT FOUND (will be NaN)")
        elif len(found) == 1 and found[0][0] == standard_name:
            print(f"  ✓ {standard_name:<25} — same name")
        else:
            for name, label in found:
                rename = f"→ rename to '{standard_name}'" if name != standard_name else "same name"
                print(f"  ~ {standard_name:<25} — found as '{name}' ({rename})")
                print(f"    label: {label[:80]}")


Y3_2017-18
  ~ VolAny                    — found as 'VOLANY' (→ rename to 'VolAny')
    label: Whether done any volunteering in last 12 months (includes raised funds)
  ~ VolAny                    — found as 'Volint_ANY' (→ rename to 'VolAny')
    label: Whether done any volunteering in last 12 months
  ~ VolFrqB_Pop               — found as 'VOLFRQ_POP' (→ rename to 'VolFrqB_Pop')
    label: Volunteered at least 2x in the last 12 months excluding those doing solely raisi
  ~ VolCnt_GR2                — found as 'Volcnt_GR2' (→ rename to 'VolCnt_GR2')
    label: one vs two or more roles
  ✓ VolCnt                    — same name
  ~ volint1_vol               — found as 'volint1' (→ rename to 'volint1_vol')
    label: During the last 12 months, have you given any of your time to do any of the foll
  ~ volint1_vol               — found as 'volint1_just' (→ rename to 'volint1_vol')
    label: Just this role: Raise funds for a sports club, organisation or event
  ~ volint2_vol             

In [2]:
import pyreadstat

SAV_Y3 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav"
_, meta_y3 = pyreadstat.read_sav(SAV_Y3, metadataonly=True)

la_cols = [v for v in meta_y3.column_names if v.upper().startswith('LA_')]
print(f"LA columns in Y3: {la_cols}")

LA columns in Y3: ['LA_2023', 'LA_2021', 'LA_2020', 'LA_Pre2019', 'LA_Old']


In [3]:
# Check which LA column has London E09 codes
for la_col in ['LA_Pre2019', 'LA_Old', 'LA_2020', 'LA_2021', 'LA_2023']:
    labels = meta_y3.variable_value_labels.get(la_col, {})
    london = {code: name for code, name in labels.items() 
              if 'E09' in str(name) and 'E09000001' not in str(name)}
    print(f"\n{la_col}: {len(london)} London boroughs found")
    if london:
        # Show first 3 as sample
        for code, name in list(london.items())[:3]:
            print(f"  {code}: {name}")
            


LA_Pre2019: 32 London boroughs found
  9.0: E09000002 Barking and Dagenham
  10.0: E09000003 Barnet
  18.0: E09000004 Bexley

LA_Old: 0 London boroughs found

LA_2020: 32 London boroughs found
  8.0: E09000002 Barking and Dagenham
  9.0: E09000003 Barnet
  17.0: E09000004 Bexley

LA_2021: 32 London boroughs found
  8.0: E09000002 Barking and Dagenham
  9.0: E09000003 Barnet
  17.0: E09000004 Bexley

LA_2023: 32 London boroughs found
  8.0: E09000002 Barking and Dagenham
  9.0: E09000003 Barnet
  17.0: E09000004 Bexley


In [5]:
#final data for year 3
import pyreadstat
import pandas as pd

SAV_Y3 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav"

# ── Step 1: Load metadata ─────────────────────────────────────────────────────
_, meta_y3 = pyreadstat.read_sav(SAV_Y3, metadataonly=True)
file_vars_y3 = set(meta_y3.column_names)

# ── Step 2: Define columns ────────────────────────────────────────────────────
core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
        'MonthTwelve', 'group', 'mode']
geo  = [v for v in file_vars_y3 if v.startswith('LA_')]

p1_prefixes = [
    'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
    'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
    'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
    'DURATION', 'DURATIONGR', 'MINS_SESS',
    'ACTYRA', 'ACTYRB', 'ACTYRC',
    'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
]
p1_exact_names = [
    'MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_SPORTCOUNT_A01',
    'MEMS7GR_SPORTCOUNT_A01', 'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
    'Number_Activities_150', 'Number_Activities_Gr2', 'Number_Activities_Gr5',
    'Number_Activities_150_Gr2', 'Number_Activities_150_Gr5', 'DAYS10P60GR',
]
p1_exact     = [v for v in file_vars_y3 if v in p1_exact_names or v in p1_prefixes]
p1_composite = [v for v in file_vars_y3 if
                any(v.startswith(p) for p in p1_prefixes) and
                any(s in v for s in ['_C0', '_B0', '_D0']) and
                'GARD' not in v.upper()]
club_summary = [v for v in file_vars_y3 if
                v.startswith('CLUB_') and
                any(v.endswith(s) for s in [
                    '_A01','_A02','_A03','_B01','_B03','_B05','_B06','_B07',
                    '_C01','_C02','_C04','_C05','_C06','_C07','_C08','_C09',
                    '_C10','_C11','_C13','_C14','_C15'
                ])]
club_summary += [v for v in ['CLUB_SPORTCOUNT_A01','Number_Club','Number_Club_Gr2',
                              'Club_ExcFitness','CLUB_SPORTFUND_A02'] if v in file_vars_y3]
barriers = [v for v in file_vars_y3 if v.startswith('limfreti')]
p1       = list(set(p1_exact + p1_composite + club_summary + barriers))

demo = [v for v in [
    'Age9', 'Age5_2', 'Gend3', 'GendAge9',
    'Eth7', 'Eth2', 'EthAge4',
    'Disab3', 'Disab2_POP', 'NSSEC5', 'NSSEC8', 'Educ6',
] if v in file_vars_y3]

health = [v for v in [
    'health', 'BMIG',
    'disty1_POP','disty2_POP','disty3_POP','disty4_POP',
    'disty5_POP','disty6_POP','disty7_POP','disty8_POP',
    'disty9_POP','disty10_POP',
] if v in file_vars_y3]

attitudes = [v for v in [
    'READYAB1_POP', 'READYOP1_POP',
    'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
] if v in file_vars_y3]

p2 = list(set(demo + health + attitudes))

mems_inout = [v for v in file_vars_y3 if
              any(v.startswith(p) for p in ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
              'SPORTCOUNT_A01' in v]
inclus     = [v for v in file_vars_y3 if
              v.lower().startswith('inclus') and 'GR2' not in v]
p3         = list(set(mems_inout + inclus))

# Y3 specific P4 — use volint1-7 (not _vol), VOLANY, VOLFRQ_POP
p4 = [v for v in [
    'VOLANY', 'volint1', 'volint2', 'volint3', 'volint4',
    'volint5', 'volint6', 'volint7',
    'VOLFRQ_POP', 'VolCnt', 'Volcnt_GR2',
] if v in file_vars_y3]

# ── Step 3: Combine and validate ──────────────────────────────────────────────
la_col     = 'LA_Pre2019'
all_cols   = list(set(core + geo + p1 + p2 + p3 + p4))
valid_cols = [v for v in all_cols if v in file_vars_y3]

# Ensure LA column is included
if la_col not in valid_cols:
    valid_cols.append(la_col)

print(f"Core:    {len([v for v in core if v in file_vars_y3])}")
print(f"Geo:     {len(geo)}")
print(f"P1:      {len(p1)}")
print(f"P2:      {len(p2)}")
print(f"P3:      {len(p3)}")
print(f"P4:      {len(p4)}")
print(f"Total:   {len(valid_cols)}")

# ── Step 4: Load data ─────────────────────────────────────────────────────────
print("\nLoading data...")
df_y3, _ = pyreadstat.read_sav(SAV_Y3, usecols=valid_cols, apply_value_formats=False)
print(f"Full shape: {df_y3.shape}")

# ── Step 5: Rename to standard names ─────────────────────────────────────────
rename_map = {
    'VOLANY':     'VolAny',
    'volint1':    'volint1_vol',
    'volint2':    'volint2_vol',
    'volint3':    'volint3_vol',
    'volint4':    'volint4_vol',
    'volint5':    'volint5_vol',
    'volint6':    'volint6_vol',
    'volint7':    'volint7_vol',
    'VOLFRQ_POP': 'VolFrqB_Pop',
    'Volcnt_GR2': 'VolCnt_GR2',
}
df_y3 = df_y3.rename(columns=rename_map)
print(f"Renamed {len(rename_map)} columns ✓")

# ── Step 6: Filter to London (excl. City of London) ──────────────────────────
borough_labels  = meta_y3.variable_value_labels.get(la_col, {})
london_boroughs = {code: name for code, name in borough_labels.items()
                   if 'E09' in str(name) and 'E09000001' not in str(name)}
london_codes    = list(london_boroughs.keys())

df_y3_london = df_y3[df_y3[la_col].isin(london_codes)].copy()

# ── Step 7: Standardise and save ─────────────────────────────────────────────
df_y3_london['survey_year'] = '2017-18'
df_y3_london = df_y3_london.rename(columns={la_col: 'LA_code'})

print(f"\nLondon boroughs: {len(london_boroughs)}")
print(f"London rows:     {df_y3_london.shape[0]:,}")
print(f"Final shape:     {df_y3_london.shape}")

df_y3_london.to_csv('active_lives_london_y3_2017-18.csv', index=False)
print("\n✓ Saved: active_lives_london_y3_2017-18.csv")

Core:    7
Geo:     5
P1:      455
P2:      29
P3:      19
P4:      11
Total:   526

Loading data...
Full shape: (179747, 526)
Renamed 10 columns ✓

London boroughs: 32
London rows:     15,967
Final shape:     (15967, 527)

✓ Saved: active_lives_london_y3_2017-18.csv


In [6]:
import pyreadstat

SAV_Y2 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav"
_, meta_y2 = pyreadstat.read_sav(SAV_Y2, metadataonly=True)

la_cols = [v for v in meta_y2.column_names if v.upper().startswith('LA_')]
print(f"LA columns in Y2: {la_cols}")

# Check which ones have London E09 codes
for la_col in la_cols:
    labels = meta_y2.variable_value_labels.get(la_col, {})
    london = {code: name for code, name in labels.items()
              if 'E09' in str(name) and 'E09000001' not in str(name)}
    print(f"  {la_col}: {len(london)} London boroughs")

LA columns in Y2: ['LA_2020', 'LA_2021', 'LA_2023', 'LA_Old']
  LA_2020: 32 London boroughs
  LA_2021: 32 London boroughs
  LA_2023: 32 London boroughs
  LA_Old: 0 London boroughs


In [7]:
#final data for year 2
import pyreadstat
import pandas as pd

SAV_Y2 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav"

# ── Step 1: Load metadata ─────────────────────────────────────────────────────
_, meta_y2 = pyreadstat.read_sav(SAV_Y2, metadataonly=True)
file_vars_y2 = set(meta_y2.column_names)

# ── Step 2: Define columns ────────────────────────────────────────────────────
core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
        'MonthTwelve', 'group', 'mode']
geo  = [v for v in file_vars_y2 if v.startswith('LA_')]

p1_prefixes = [
    'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
    'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
    'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
    'DURATION', 'DURATIONGR', 'MINS_SESS',
    'ACTYRA', 'ACTYRB', 'ACTYRC',
    'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
]
p1_exact_names = [
    'MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_SPORTCOUNT_A01',
    'MEMS7GR_SPORTCOUNT_A01', 'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
    'Number_Activities_150', 'Number_Activities_Gr2', 'Number_Activities_Gr5',
    'Number_Activities_150_Gr2', 'Number_Activities_150_Gr5', 'DAYS10P60GR',
]
p1_exact     = [v for v in file_vars_y2 if v in p1_exact_names or v in p1_prefixes]
p1_composite = [v for v in file_vars_y2 if
                any(v.startswith(p) for p in p1_prefixes) and
                any(s in v for s in ['_C0', '_B0', '_D0']) and
                'GARD' not in v.upper()]
club_summary = [v for v in file_vars_y2 if
                v.startswith('CLUB_') and
                any(v.endswith(s) for s in [
                    '_A01','_A02','_A03','_B01','_B03','_B05','_B06','_B07',
                    '_C01','_C02','_C04','_C05','_C06','_C07','_C08','_C09',
                    '_C10','_C11','_C13','_C14','_C15'
                ])]
club_summary += [v for v in ['CLUB_SPORTCOUNT_A01','Number_Club','Number_Club_Gr2',
                              'Club_ExcFitness','CLUB_SPORTFUND_A02'] if v in file_vars_y2]
barriers = [v for v in file_vars_y2 if v.startswith('limfreti')]
p1       = list(set(p1_exact + p1_composite + club_summary + barriers))

demo = [v for v in [
    'Age9', 'Age5_2', 'Gend3', 'GendAge9',
    'Eth7', 'Eth2', 'EthAge4',
    'Disab3', 'Disab2_POP', 'NSSEC5', 'NSSEC8', 'Educ6',
] if v in file_vars_y2]

health = [v for v in [
    'health', 'BMIG',
    'disty1_POP','disty2_POP','disty3_POP','disty4_POP',
    'disty5_POP','disty6_POP','disty7_POP','disty8_POP',
    'disty9_POP','disty10_POP',
] if v in file_vars_y2]

attitudes = [v for v in [
    'READYAB1_POP', 'READYOP1_POP',
    'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
] if v in file_vars_y2]

p2 = list(set(demo + health + attitudes))

mems_inout = [v for v in file_vars_y2 if
              any(v.startswith(p) for p in ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
              'SPORTCOUNT_A01' in v]
inclus     = [v for v in file_vars_y2 if
              v.lower().startswith('inclus') and 'GR2' not in v]
p3         = list(set(mems_inout + inclus))

# Y2 specific P4
p4 = [v for v in [
    'VOLANY', 'volint1', 'volint2', 'volint3', 'volint4',
    'volint5', 'volint6', 'volint7',
    'VOLFRQ_POP', 'VolCnt', 'Volcnt_GR2',
] if v in file_vars_y2]

# ── Step 3: Combine and validate ──────────────────────────────────────────────
la_col     = 'LA_2020'
all_cols   = list(set(core + geo + p1 + p2 + p3 + p4))
valid_cols = [v for v in all_cols if v in file_vars_y2]

# Ensure LA column is included
if la_col not in valid_cols:
    valid_cols.append(la_col)

print(f"Core:    {len([v for v in core if v in file_vars_y2])}")
print(f"Geo:     {len(geo)}")
print(f"P1:      {len(p1)}")
print(f"P2:      {len(p2)}")
print(f"P3:      {len(p3)}")
print(f"P4:      {len(p4)}")
print(f"Total:   {len(valid_cols)}")

# ── Step 4: Load data ─────────────────────────────────────────────────────────
print("\nLoading data...")
df_y2, _ = pyreadstat.read_sav(SAV_Y2, usecols=valid_cols, apply_value_formats=False)
print(f"Full shape: {df_y2.shape}")

# ── Step 5: Rename to standard names ─────────────────────────────────────────
rename_map = {
    'VOLANY':     'VolAny',
    'volint1':    'volint1_vol',
    'volint2':    'volint2_vol',
    'volint3':    'volint3_vol',
    'volint4':    'volint4_vol',
    'volint5':    'volint5_vol',
    'volint6':    'volint6_vol',
    'volint7':    'volint7_vol',
    'VOLFRQ_POP': 'VolFrqB_Pop',
    'Volcnt_GR2': 'VolCnt_GR2',
}
df_y2 = df_y2.rename(columns=rename_map)
print(f"Renamed {len(rename_map)} columns ✓")

# ── Step 6: Filter to London (excl. City of London) ──────────────────────────
borough_labels  = meta_y2.variable_value_labels.get(la_col, {})
london_boroughs = {code: name for code, name in borough_labels.items()
                   if 'E09' in str(name) and 'E09000001' not in str(name)}
london_codes    = list(london_boroughs.keys())

df_y2_london = df_y2[df_y2[la_col].isin(london_codes)].copy()

# ── Step 7: Standardise and save ─────────────────────────────────────────────
df_y2_london['survey_year'] = '2016-17'
df_y2_london = df_y2_london.rename(columns={la_col: 'LA_code'})

print(f"\nLondon boroughs: {len(london_boroughs)}")
print(f"London rows:     {df_y2_london.shape[0]:,}")
print(f"Final shape:     {df_y2_london.shape}")

df_y2_london.to_csv('active_lives_london_y2_2016-17.csv', index=False)
print("\n✓ Saved: active_lives_london_y2_2016-17.csv")

Core:    8
Geo:     4
P1:      426
P2:      29
P3:      19
P4:      11
Total:   497

Loading data...
Full shape: (196635, 497)
Renamed 10 columns ✓

London boroughs: 32
London rows:     19,248
Final shape:     (19248, 498)

✓ Saved: active_lives_london_y2_2016-17.csv


In [8]:
import pyreadstat

SAV_Y1 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"
_, meta_y1 = pyreadstat.read_sav(SAV_Y1, metadataonly=True)

la_cols = [v for v in meta_y1.column_names if v.upper().startswith('LA_')]
print(f"LA columns in Y1: {la_cols}")

# Check which have London E09 codes
for la_col in la_cols:
    labels = meta_y1.variable_value_labels.get(la_col, {})
    london = {code: name for code, name in labels.items()
              if 'E09' in str(name) and 'E09000001' not in str(name)}
    print(f"  {la_col}: {len(london)} London boroughs")

LA columns in Y1: ['LA_2021', 'LA_2023', 'LA_Old']
  LA_2021: 32 London boroughs
  LA_2023: 32 London boroughs
  LA_Old: 0 London boroughs


In [9]:
import pyreadstat
import pandas as pd

SAV_Y1 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"

# ── Step 1: Load metadata ─────────────────────────────────────────────────────
_, meta_y1 = pyreadstat.read_sav(SAV_Y1, metadataonly=True)
file_vars_y1 = set(meta_y1.column_names)

# ── Step 2: Define columns ────────────────────────────────────────────────────
core = ['serial', 'wt_final', 'xStrata', 'Quarter', 'month',
        'MonthTwelve', 'group', 'mode']
geo  = [v for v in file_vars_y1 if v.startswith('LA_')]

p1_prefixes = [
    'MEMS7', 'MEMS7GR_ALL', 'MUSCLE7', 'MUSCLE7GR',
    'DAYS10P', 'DAYS10P60', 'DAYS10P60GR',
    'MONTHS_12', 'FREQUENCY', 'FREQUENCYGR',
    'DURATION', 'DURATIONGR', 'MINS_SESS',
    'ACTYRA', 'ACTYRB', 'ACTYRC',
    'DUR_LHT', 'DUR_MOD', 'DUR_HVY', 'DURATION1PL'
]
p1_exact_names = [
    'MEMS7_ALL', 'MEMS7GR_ALL', 'MEMS7_SPORTCOUNT_A01',
    'MEMS7GR_SPORTCOUNT_A01', 'MEMS7GR7_SPORTCOUNT_A01', 'MUSCLEA',
    'Number_Activities_150', 'Number_Activities_Gr2', 'Number_Activities_Gr5',
    'Number_Activities_150_Gr2', 'Number_Activities_150_Gr5', 'DAYS10P60GR',
]
p1_exact     = [v for v in file_vars_y1 if v in p1_exact_names or v in p1_prefixes]
p1_composite = [v for v in file_vars_y1 if
                any(v.startswith(p) for p in p1_prefixes) and
                any(s in v for s in ['_C0', '_B0', '_D0']) and
                'GARD' not in v.upper()]
club_summary = [v for v in file_vars_y1 if
                v.startswith('CLUB_') and
                any(v.endswith(s) for s in [
                    '_A01','_A02','_A03','_B01','_B03','_B05','_B06','_B07',
                    '_C01','_C02','_C04','_C05','_C06','_C07','_C08','_C09',
                    '_C10','_C11','_C13','_C14','_C15'
                ])]
club_summary += [v for v in ['CLUB_SPORTCOUNT_A01','Number_Club','Number_Club_Gr2',
                              'Club_ExcFitness','CLUB_SPORTFUND_A02'] if v in file_vars_y1]
barriers = [v for v in file_vars_y1 if v.startswith('limfreti')]
p1       = list(set(p1_exact + p1_composite + club_summary + barriers))

demo = [v for v in [
    'Age9', 'Age5_2', 'Gend3', 'GendAge9',
    'Eth7', 'Eth2', 'EthAge4',
    'Disab3', 'Disab2_POP', 'NSSEC5', 'NSSEC8', 'Educ6',
] if v in file_vars_y1]

health = [v for v in [
    'health', 'BMIG',
    'disty1_POP','disty2_POP','disty3_POP','disty4_POP',
    'disty5_POP','disty6_POP','disty7_POP','disty8_POP',
    'disty9_POP','disty10_POP',
] if v in file_vars_y1]

attitudes = [v for v in [
    'READYAB1_POP', 'READYOP1_POP',
    'Motiva_POP', 'motivb_POP', 'motivc_POP', 'motivd_POP',
] if v in file_vars_y1]

p2 = list(set(demo + health + attitudes))

mems_inout = [v for v in file_vars_y1 if
              any(v.startswith(p) for p in ['MEMS7GR_IN_', 'MEMS7GR_OUT_']) and
              'SPORTCOUNT_A01' in v]
inclus     = [v for v in file_vars_y1 if
              v.lower().startswith('inclus') and 'GR2' not in v]
p3         = list(set(mems_inout + inclus))

# Y1 — P4 completely absent, no volunteering variables
p4 = []

# ── Step 3: Combine and validate ──────────────────────────────────────────────
la_col     = 'LA_2021'
all_cols   = list(set(core + geo + p1 + p2 + p3 + p4))
valid_cols = [v for v in all_cols if v in file_vars_y1]

# Ensure LA column is included
if la_col not in valid_cols:
    valid_cols.append(la_col)

print(f"Core:    {len([v for v in core if v in file_vars_y1])}")
print(f"Geo:     {len(geo)}")
print(f"P1:      {len(p1)}")
print(f"P2:      {len(p2)}")
print(f"P3:      {len(p3)}")
print(f"P4:      {len(p4)} (volunteering absent in Y1)")
print(f"Total:   {len(valid_cols)}")

# ── Step 4: Load data ─────────────────────────────────────────────────────────
print("\nLoading data...")
df_y1, _ = pyreadstat.read_sav(SAV_Y1, usecols=valid_cols, apply_value_formats=False)
print(f"Full shape: {df_y1.shape}")

# ── Step 5: No renames needed for Y1 ─────────────────────────────────────────
print("No renames needed for Y1 ✓")

# ── Step 6: Filter to London (excl. City of London) ──────────────────────────
borough_labels  = meta_y1.variable_value_labels.get(la_col, {})
london_boroughs = {code: name for code, name in borough_labels.items()
                   if 'E09' in str(name) and 'E09000001' not in str(name)}
london_codes    = list(london_boroughs.keys())

df_y1_london = df_y1[df_y1[la_col].isin(london_codes)].copy()

# ── Step 7: Standardise and save ─────────────────────────────────────────────
df_y1_london['survey_year'] = '2015-16'
df_y1_london = df_y1_london.rename(columns={la_col: 'LA_code'})

print(f"\nLondon boroughs: {len(london_boroughs)}")
print(f"London rows:     {df_y1_london.shape[0]:,}")
print(f"Final shape:     {df_y1_london.shape}")

df_y1_london.to_csv('active_lives_london_y1_2015-16.csv', index=False)
print("\n✓ Saved: active_lives_london_y1_2015-16.csv")

Core:    7
Geo:     3
P1:      258
P2:      23
P3:      0
P4:      0 (volunteering absent in Y1)
Total:   291

Loading data...
Full shape: (198911, 291)
No renames needed for Y1 ✓

London boroughs: 32
London rows:     19,620
Final shape:     (19620, 292)

✓ Saved: active_lives_london_y1_2015-16.csv


In [10]:
import pandas as pd
import glob
import os

# Load all 8 yearly files
files = {
    'y1': 'active_lives_london_y1_2015-16.csv',
    'y2': 'active_lives_london_y2_2016-17.csv',
    'y3': 'active_lives_london_y3_2017-18.csv',
    'y4': 'active_lives_london_y4_2018-19.csv',
    'y5': 'active_lives_london_y5_2019-20.csv',
    'y6': 'active_lives_london_y6_2020-21.csv',
    'y7': 'active_lives_london_y7_2021-22.csv',
    'y8': 'active_lives_london_y8_2022-23.csv',
}

dfs = {}
for label, fname in files.items():
    dfs[label] = pd.read_csv(fname, low_memory=False)
    print(f"{label}: {dfs[label].shape}")

# Stack all years
df_all = pd.concat(dfs.values(), ignore_index=True)
print(f"\nMerged dataset: {df_all.shape}")
print(f"Survey years:   {sorted(df_all['survey_year'].unique())}")
print(f"LA codes:       {df_all['LA_code'].nunique()} unique boroughs")

y1: (19620, 292)
y2: (19248, 498)
y3: (15967, 527)
y4: (15889, 545)
y5: (16091, 589)
y6: (16028, 589)
y7: (16139, 590)
y8: (16515, 601)

Merged dataset: (135497, 609)
Survey years:   ['2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
LA codes:       78 unique boroughs


In [11]:
# Check what LA codes look like per year
for label, df in dfs.items():
    n_boroughs = df['LA_code'].nunique()
    sample = sorted(df['LA_code'].dropna().unique())[:5]
    print(f"{label}: {n_boroughs} unique codes — sample: {sample}")

y1: 32 unique codes — sample: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0)]
y2: 32 unique codes — sample: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0)]
y3: 32 unique codes — sample: [np.float64(9.0), np.float64(10.0), np.float64(18.0), np.float64(31.0), np.float64(36.0)]
y4: 32 unique codes — sample: [np.float64(9.0), np.float64(10.0), np.float64(18.0), np.float64(31.0), np.float64(36.0)]
y5: 32 unique codes — sample: [np.float64(9.0), np.float64(10.0), np.float64(18.0), np.float64(31.0), np.float64(36.0)]
y6: 32 unique codes — sample: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0)]
y7: 32 unique codes — sample: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0)]
y8: 32 unique codes — sample: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0)]


In [12]:
import pyreadstat

# Load metadata for one file from each coding scheme
# Scheme A: Y1/Y2/Y6/Y7/Y8 — codes like 8.0, 9.0, 17.0
# Scheme B: Y3/Y4/Y5 — codes like 9.0, 10.0, 18.0

SAV_SCHEME_A = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav"
SAV_SCHEME_B = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav"

# Get borough name labels for each scheme
_, meta_a = pyreadstat.read_sav(SAV_SCHEME_A, metadataonly=True)
_, meta_b = pyreadstat.read_sav(SAV_SCHEME_B, metadataonly=True)

# Scheme A uses LA_2023, Scheme B uses LA_2019
labels_a = meta_a.variable_value_labels.get('LA_2023', {})
labels_b = meta_b.variable_value_labels.get('LA_2019', {})

# Build lookup: numeric code → clean borough name (E09 London only, excl City)
def build_lookup(labels):
    return {
        code: name.split(' ', 1)[1]  # remove E09XXXXXX prefix, keep name only
        for code, name in labels.items()
        if 'E09' in str(name) and 'E09000001' not in str(name)
    }

lookup_a = build_lookup(labels_a)
lookup_b = build_lookup(labels_b)

print("Scheme A sample (Y1/Y2/Y6/Y7/Y8):")
for k, v in sorted(lookup_a.items())[:5]:
    print(f"  {k} → {v}")

print("\nScheme B sample (Y3/Y4/Y5):")
for k, v in sorted(lookup_b.items())[:5]:
    print(f"  {k} → {v}")

print(f"\nScheme A boroughs: {len(lookup_a)}")
print(f"Scheme B boroughs: {len(lookup_b)}")

Scheme A sample (Y1/Y2/Y6/Y7/Y8):
  8.0 → Barking and Dagenham
  9.0 → Barnet
  17.0 → Bexley
  30.0 → Brent
  35.0 → Bromley

Scheme B sample (Y3/Y4/Y5):
  9.0 → Barking and Dagenham
  10.0 → Barnet
  18.0 → Bexley
  31.0 → Brent
  36.0 → Bromley

Scheme A boroughs: 32
Scheme B boroughs: 32


In [13]:
# Define which years use which scheme
scheme_a_years = ['2015-16', '2016-17', '2020-21', '2021-22', '2022-23']
scheme_b_years = ['2017-18', '2018-19', '2019-20']

# Apply correct lookup to each yearly dataframe
for label, df in dfs.items():
    year = df['survey_year'].iloc[0]
    if year in scheme_a_years:
        df['borough'] = df['LA_code'].map(lookup_a)
    else:
        df['borough'] = df['LA_code'].map(lookup_b)

# Restack with borough column
df_all = pd.concat(dfs.values(), ignore_index=True)

# Verify
print(f"Merged shape: {df_all.shape}")
print(f"Unique boroughs: {df_all['borough'].nunique()}")
print(f"Borough names: {sorted(df_all['borough'].dropna().unique())}")
print(f"\nAny unmapped rows: {df_all['borough'].isna().sum()}")

# Check each year has 32 boroughs
print("\nBoroughs per year:")
print(df_all.groupby('survey_year')['borough'].nunique())

Merged shape: (135497, 610)
Unique boroughs: 32
Borough names: ['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster']

Any unmapped rows: 45081

Boroughs per year:
survey_year
2015-16    11
2016-17    11
2017-18    32
2018-19    32
2019-20    32
2020-21    11
2021-22    11
2022-23    32
Name: borough, dtype: int64


In [14]:
# Check which codes in Y1 aren't in lookup_a
y1_codes = set(dfs['y1']['LA_code'].dropna().unique())
lookup_a_codes = set(lookup_a.keys())

missing_from_lookup = y1_codes - lookup_a_codes
print(f"Y1 codes not in Scheme A lookup: {sorted(missing_from_lookup)}")
print(f"\nY1 all codes: {sorted(y1_codes)}")
print(f"Lookup A codes: {sorted(lookup_a_codes)}")


Y1 codes not in Scheme A lookup: [np.float64(69.0), np.float64(80.0), np.float64(94.0), np.float64(110.0), np.float64(115.0), np.float64(120.0), np.float64(125.0), np.float64(132.0), np.float64(138.0), np.float64(143.0), np.float64(146.0), np.float64(151.0), np.float64(164.0), np.float64(175.0), np.float64(206.0), np.float64(247.0), np.float64(261.0), np.float64(278.0), np.float64(285.0), np.float64(286.0), np.float64(300.0)]

Y1 all codes: [np.float64(8.0), np.float64(9.0), np.float64(17.0), np.float64(30.0), np.float64(35.0), np.float64(44.0), np.float64(69.0), np.float64(80.0), np.float64(94.0), np.float64(110.0), np.float64(112.0), np.float64(115.0), np.float64(117.0), np.float64(120.0), np.float64(125.0), np.float64(129.0), np.float64(132.0), np.float64(138.0), np.float64(139.0), np.float64(143.0), np.float64(146.0), np.float64(151.0), np.float64(164.0), np.float64(175.0), np.float64(201.0), np.float64(206.0), np.float64(247.0), np.float64(261.0), np.float64(278.0), np.float64(285

In [15]:
# Get the correct lookup for Y1's LA_2021 column
SAV_Y1 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"
SAV_Y2 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav"
SAV_Y6 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav"
SAV_Y7 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav"

# LA columns used per year
year_la_map = {
    'y1': (SAV_Y1, 'LA_2021'),
    'y2': (SAV_Y2, 'LA_2020'),
    'y6': (SAV_Y6, 'LA_2020'),
    'y7': (SAV_Y7, 'LA_2021'),
}

# Build a lookup per year directly from its own SPSS metadata
lookups = {}
for label, (path, la_col) in year_la_map.items():
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    labels = meta.variable_value_labels.get(la_col, {})
    lookups[label] = {
        code: name.split(' ', 1)[1]
        for code, name in labels.items()
        if 'E09' in str(name) and 'E09000001' not in str(name)
    }
    print(f"{label} ({la_col}): {len(lookups[label])} boroughs — sample: {list(lookups[label].items())[:3]}")

# Also keep scheme_b for Y3/Y4/Y5
lookups['y3'] = lookup_b
lookups['y4'] = lookup_b
lookups['y5'] = lookup_b

# Y6/Y7/Y8 use scheme_a but we now have per-year lookups for Y6/Y7
lookups['y8'] = lookup_a

y1 (LA_2021): 32 boroughs — sample: [(8.0, 'Barking and Dagenham'), (9.0, 'Barnet'), (17.0, 'Bexley')]
y2 (LA_2020): 32 boroughs — sample: [(8.0, 'Barking and Dagenham'), (9.0, 'Barnet'), (17.0, 'Bexley')]
y6 (LA_2020): 32 boroughs — sample: [(8.0, 'Barking and Dagenham'), (9.0, 'Barnet'), (17.0, 'Bexley')]
y7 (LA_2021): 32 boroughs — sample: [(8.0, 'Barking and Dagenham'), (9.0, 'Barnet'), (17.0, 'Bexley')]


In [16]:
# Apply correct lookup to each year
year_lookup_map = {
    'y1': lookups['y1'],
    'y2': lookups['y2'],
    'y3': lookups['y3'],
    'y4': lookups['y4'],
    'y5': lookups['y5'],
    'y6': lookups['y6'],
    'y7': lookups['y7'],
    'y8': lookups['y8'],
}

for label, df in dfs.items():
    df['borough'] = df['LA_code'].map(year_lookup_map[label])

# Restack
df_all = pd.concat(dfs.values(), ignore_index=True)

# Verify
print(f"Merged shape:     {df_all.shape}")
print(f"Unique boroughs:  {df_all['borough'].nunique()}")
print(f"Unmapped rows:    {df_all['borough'].isna().sum()}")

print("\nBoroughs per year:")
print(df_all.groupby('survey_year')['borough'].nunique())

print("\nRows per year:")
print(df_all.groupby('survey_year')['borough'].count())

Merged shape:     (135497, 610)
Unique boroughs:  32
Unmapped rows:    0

Boroughs per year:
survey_year
2015-16    32
2016-17    32
2017-18    32
2018-19    32
2019-20    32
2020-21    32
2021-22    32
2022-23    32
Name: borough, dtype: int64

Rows per year:
survey_year
2015-16    19620
2016-17    19248
2017-18    15967
2018-19    15889
2019-20    16091
2020-21    16028
2021-22    16139
2022-23    16515
Name: borough, dtype: int64


In [17]:
# Save merged dataset
df_all.to_csv('active_lives_london_all_years.csv', index=False)
print("✓ Saved: active_lives_london_all_years.csv")
print(f"  Shape: {df_all.shape}")
print(f"  Years: 2015-16 to 2022-23")
print(f"  Boroughs: 32")
print(f"  Total respondents: {len(df_all):,}")

✓ Saved: active_lives_london_all_years.csv
  Shape: (135497, 610)
  Years: 2015-16 to 2022-23
  Boroughs: 32
  Total respondents: 135,497


In [18]:
# Known sentinel values in Active Lives Survey
SENTINELS = [-99, -98, -97, -96, -95, -94]

# Check all numeric columns for sentinel contamination
numeric_cols = df_all.select_dtypes(include='number').columns.tolist()

sentinel_report = []
for col in numeric_cols:
    col_vals = df_all[col].dropna()
    for s in SENTINELS:
        count = (col_vals == s).sum()
        if count > 0:
            pct = count / len(df_all) * 100
            sentinel_report.append({
                'column': col,
                'sentinel': s,
                'count': count,
                'pct_of_total': round(pct, 2)
            })

sentinel_df = pd.DataFrame(sentinel_report)

print(f"Columns with sentinel values: {sentinel_df['column'].nunique()}")
print(f"Total sentinel entries found: {sentinel_df['count'].sum():,}")

# Show worst offenders
print("\nTop 20 most affected columns:")
top = sentinel_df.groupby('column')['count'].sum().sort_values(ascending=False).head(20)
print(top.to_string())

# Show which sentinels are most common
print("\nSentinel value frequency:")
print(sentinel_df.groupby('sentinel')['count'].sum().sort_values(ascending=False))

KeyError: 'column'

In [19]:
# Check if sentinels exist at all
print("Quick sentinel check on key variables:")
key_vars = ['Age9', 'Gend3', 'MEMS7GR_ALL', 'VolAny', 'Eth7', 'NSSEC5']

for col in key_vars:
    if col in df_all.columns:
        vals = df_all[col].dropna().unique()
        neg_vals = [v for v in vals if v < 0]
        print(f"  {col}: negative values = {sorted(neg_vals)[:5] if neg_vals else 'none'}")
    else:
        print(f"  {col}: NOT IN DATASET")

# Also check the raw min values across all numeric cols
numeric_cols = df_all.select_dtypes(include='number').columns.tolist()
mins = df_all[numeric_cols].min()
negative_cols = mins[mins < -90]
print(f"\nColumns with values below -90: {len(negative_cols)}")
print(negative_cols.head(10))

Quick sentinel check on key variables:
  Age9: negative values = none
  Gend3: negative values = none
  MEMS7GR_ALL: negative values = none
  VolAny: negative values = none
  Eth7: negative values = none
  NSSEC5: negative values = none

Columns with values below -90: 0
Series([], dtype: float64)


In [20]:
# Confirm - check what the actual minimum values look like
numeric_cols = df_all.select_dtypes(include='number').columns.tolist()
mins = df_all[numeric_cols].min()

print(f"Total numeric columns: {len(numeric_cols)}")
print(f"Minimum value across all columns: {mins.min()}")
print(f"Columns with any negative values: {(mins < 0).sum()}")

# Check NaN rates on key variables to confirm sentinels became NaN
print("\nNaN rates on key variables:")
key_vars = ['Age9', 'Gend3', 'MEMS7GR_ALL', 'VolAny', 
            'Eth7', 'NSSEC5', 'health', 'READYOP1_POP']
for col in key_vars:
    if col in df_all.columns:
        nan_pct = df_all[col].isna().mean() * 100
        print(f"  {col}: {nan_pct:.1f}% NaN")

Total numeric columns: 608
Minimum value across all columns: -1.0
Columns with any negative values: 1

NaN rates on key variables:
  Age9: 1.0% NaN
  Gend3: 0.2% NaN
  MEMS7GR_ALL: 0.0% NaN
  VolAny: 37.1% NaN
  Eth7: 6.8% NaN
  NSSEC5: 6.4% NaN
  health: 41.7% NaN
  READYOP1_POP: 49.6% NaN


In [21]:
# Find the column with negative values
numeric_cols = df_all.select_dtypes(include='number').columns.tolist()
mins = df_all[numeric_cols].min()
neg_col = mins[mins < 0]
print(f"Column with negative values: {neg_col}")

# Check its distribution
col_name = neg_col.index[0]
print(f"\nValue counts for {col_name}:")
print(df_all[col_name].value_counts().head(10))

Column with negative values: DURATION1PL_TEAMSPORT_C05   -1.0
dtype: float64

Value counts for DURATION1PL_TEAMSPORT_C05:
DURATION1PL_TEAMSPORT_C05
 0.0    76679
-1.0    17920
 3.0     2575
 4.0     2463
 2.0     1515
 1.0     1187
Name: count, dtype: int64


In [22]:
# Check the value labels for this variable from Y8 metadata
import pyreadstat

SAV_Y8 = r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav"
_, meta_y8 = pyreadstat.read_sav(SAV_Y8, metadataonly=True)

labels = meta_y8.variable_value_labels.get('DURATION1PL_TEAMSPORT_C05', {})
print("Value labels for DURATION1PL_TEAMSPORT_C05:")
for code, label in sorted(labels.items()):
    print(f"  {code}: {label}")

Value labels for DURATION1PL_TEAMSPORT_C05:
  -99.0: Missing, should have been answered
  -98.0: Not applicable: Survey routing
  -97.0: Incorrectly multicoded
  -96.0: Out of range
  -95.0: Cannot give an estimate / Don't know
  -94.0: Prefer not to say
  1.0: 1-29
  2.0: 30-59
  3.0: 60-149
  4.0: 150+


In [23]:
# Check if -1.0 appears in other DURATION1PL variables
dur1pl_cols = [c for c in df_all.columns if c.startswith('DURATION1PL_')]

print(f"Total DURATION1PL_ columns: {len(dur1pl_cols)}")
print(f"\nColumns with -1.0 values:")
for col in dur1pl_cols:
    neg_count = (df_all[col] == -1.0).sum()
    if neg_count > 0:
        print(f"  {col}: {neg_count:,} rows")

# Also check which years the -1.0 appears in
print(f"\n-1.0 in DURATION1PL_TEAMSPORT_C05 by year:")
mask = df_all['DURATION1PL_TEAMSPORT_C05'] == -1.0
print(df_all[mask].groupby('survey_year').size())

Total DURATION1PL_ columns: 19

Columns with -1.0 values:
  DURATION1PL_TEAMSPORT_C05: 17,920 rows

-1.0 in DURATION1PL_TEAMSPORT_C05 by year:
survey_year
2016-17    17920
dtype: int64


In [24]:
# Replace -1.0 with NaN in this specific column
df_all['DURATION1PL_TEAMSPORT_C05'] = df_all['DURATION1PL_TEAMSPORT_C05'].replace(-1.0, float('nan'))

# Verify
neg_remaining = (df_all['DURATION1PL_TEAMSPORT_C05'] < 0).sum()
print(f"Negative values remaining: {neg_remaining}")

# Confirm no other negative values anywhere
numeric_cols = df_all.select_dtypes(include='number').columns.tolist()
mins = df_all[numeric_cols].min()
negative_cols = mins[mins < 0]
print(f"Columns with any negative values: {len(negative_cols)}")

Negative values remaining: 0
Columns with any negative values: 0


In [25]:
# ── Step 3: Missing Value Audit ───────────────────────────────────────────────

# Define core variables for each pillar
pillar_vars = {
    'P1 - Participation': [
        'MEMS7GR_ALL', 'MEMS7GR_SPORTCOUNT_A01',
        'MONTHS_12_WALKALL_C01', 'MONTHS_12_TEAMSPORT_C05',
        'MONTHS_12_RACKETSPORT_C06', 'MONTHS_12_FITNESS_B06',
        'FREQUENCY_WALKALL_C01', 'DURATION_WALKALL_C01',
        'CLUB_SPORTCOUNT_A01', 'Number_Activities_Gr2',
        'limfreti1', 'limfreti2',
    ],
    'P2 - Demographics': [
        'Age9', 'Gend3', 'Eth7', 'Disab3',
        'NSSEC5', 'Educ6', 'health',
        'READYAB1_POP', 'READYOP1_POP',
        'Motiva_POP', 'motivb_POP',
    ],
    'P3 - Indoor/Outdoor': [
        'MEMS7GR_IN_SPORTCOUNT_A01', 'MEMS7GR_OUT_SPORTCOUNT_A01',
        'MEMS7GR_IN_HOME_SPORTCOUNT_A01',
        'MEMS7GR_OUT_LOCAL_PARK_SPORTCOUNT_A01',
        'MEMS7GR_OUT_LOCAL_ROAD_SPORTCOUNT_A01',
        'inclus_a', 'inclus_b', 'inclus_c',
    ],
    'P4 - Volunteering': [
        'VolAny', 'volint1_vol', 'volint2_vol', 'volint3_vol',
        'VolFrqB_Pop', 'VolDur_GR2', 'VolLong_GR3',
        'VolCnt', 'VolCnt_GR2',
    ],
}

print("=" * 70)
print("MISSING VALUE AUDIT — BY PILLAR AND YEAR")
print("=" * 70)

for pillar, cols in pillar_vars.items():
    print(f"\n── {pillar} ──")
    # Filter to cols that exist
    existing = [c for c in cols if c in df_all.columns]
    missing_vars = [c for c in cols if c not in df_all.columns]

    if missing_vars:
        print(f"  Not in dataset: {missing_vars}")

    # Overall NaN rate per variable
    print(f"\n  {'Variable':<40} {'Overall NaN%':>12} {'Min year NaN%':>14} {'Max year NaN%':>14}")
    print(f"  {'-'*40} {'-'*12} {'-'*14} {'-'*14}")

    for col in existing:
        overall_nan = df_all[col].isna().mean() * 100
        by_year = df_all.groupby('survey_year')[col].apply(
            lambda x: x.isna().mean() * 100
        )
        print(f"  {col:<40} {overall_nan:>11.1f}% {by_year.min():>13.1f}% {by_year.max():>13.1f}%")

MISSING VALUE AUDIT — BY PILLAR AND YEAR

── P1 - Participation ──

  Variable                                 Overall NaN%  Min year NaN%  Max year NaN%
  ---------------------------------------- ------------ -------------- --------------
  MEMS7GR_ALL                                      0.0%           0.0%           0.0%
  MEMS7GR_SPORTCOUNT_A01                           0.0%           0.0%           0.0%
  MONTHS_12_WALKALL_C01                            0.0%           0.0%           0.0%
  MONTHS_12_TEAMSPORT_C05                          0.0%           0.0%           0.0%
  MONTHS_12_RACKETSPORT_C06                        0.0%           0.0%           0.0%
  MONTHS_12_FITNESS_B06                           14.5%           0.0%         100.0%
  FREQUENCY_WALKALL_C01                            0.0%           0.0%           0.0%
  DURATION_WALKALL_C01                             0.0%           0.0%           0.0%
  CLUB_SPORTCOUNT_A01                             58.3%           0.0%  

In [26]:
# Check limfreti NaN rate specifically within Y8 only
y8_mask = df_all['survey_year'] == '2022-23'
print("limfreti NaN rates in Y8 only:")
for i in range(1, 9):
    col = f'limfreti{i}'
    if col in df_all.columns:
        nan_pct = df_all.loc[y8_mask, col].isna().mean() * 100
        val_counts = df_all.loc[y8_mask, col].value_counts()
        print(f"  {col}: {nan_pct:.1f}% NaN | values: {val_counts.to_dict()}")

limfreti NaN rates in Y8 only:
  limfreti1: 61.9% NaN | values: {0.0: 5225, 1.0: 1071}
  limfreti2: 61.9% NaN | values: {0.0: 4018, 1.0: 2278}
  limfreti3: 61.9% NaN | values: {0.0: 3528, 1.0: 2768}
  limfreti4: 61.9% NaN | values: {0.0: 6038, 1.0: 258}
  limfreti5: 61.9% NaN | values: {0.0: 5866, 1.0: 430}
  limfreti6: 61.9% NaN | values: {0.0: 4772, 1.0: 1524}
  limfreti7: 62.6% NaN | values: {0.0: 6179}
  limfreti8: 63.1% NaN | values: {0.0: 6094}


In [27]:
# Check Educ6 NaN by year
print("Educ6 NaN% by year:")
print(df_all.groupby('survey_year')['Educ6'].apply(
    lambda x: f"{x.isna().mean()*100:.1f}%"
))

Educ6 NaN% by year:
survey_year
2015-16     4.2%
2016-17     3.6%
2017-18     3.7%
2018-19     3.7%
2019-20    27.8%
2020-21     3.4%
2021-22     3.3%
2022-23     3.2%
Name: Educ6, dtype: object


In [28]:
# Check if it's concentrated in specific boroughs or quarters in Y5
y5_mask = df_all['survey_year'] == '2019-20'

print("Educ6 NaN in Y5 by Quarter:")
print(df_all[y5_mask].groupby('Quarter')['Educ6'].apply(
    lambda x: f"{x.isna().mean()*100:.1f}%"
))

print("\nEduc6 NaN in Y5 by borough (top 10 highest):")
borough_nan = df_all[y5_mask].groupby('borough')['Educ6'].apply(
    lambda x: x.isna().mean()*100
).sort_values(ascending=False)
print(borough_nan.head(10).apply(lambda x: f"{x:.1f}%"))

print("\nEduc6 NaN in Y5 by borough (top 10 lowest):")
print(borough_nan.tail(10).apply(lambda x: f"{x:.1f}%"))

Educ6 NaN in Y5 by Quarter:
Quarter
17.0    26.7%
18.0    27.2%
19.0    28.4%
20.0    29.1%
Name: Educ6, dtype: object

Educ6 NaN in Y5 by borough (top 10 highest):
borough
Havering                  36.8%
Kensington and Chelsea    35.0%
Bexley                    34.5%
Croydon                   31.5%
Barking and Dagenham      31.1%
Kingston upon Thames      30.9%
Westminster               30.7%
Bromley                   30.5%
Sutton                    30.2%
Hounslow                  29.4%
Name: Educ6, dtype: object

Educ6 NaN in Y5 by borough (top 10 lowest):
borough
Lambeth          25.9%
Southwark        25.6%
Merton           25.2%
Enfield          25.2%
Hackney          23.8%
Newham           23.8%
Hillingdon       23.5%
Camden           22.8%
Islington        22.1%
Tower Hamlets    22.0%
Name: Educ6, dtype: object


In [29]:
# ── Step 4: Borough Coverage Check ───────────────────────────────────────────

# Respondent counts per borough per year
borough_year_counts = df_all.groupby(['survey_year', 'borough']).size().unstack(fill_value=0)

print("RESPONDENTS PER BOROUGH PER YEAR")
print("=" * 80)
print(borough_year_counts.to_string())

# Summary stats per year
print("\n\nSUMMARY PER YEAR")
print("=" * 80)
print(f"{'Year':<12} {'Min':>6} {'Max':>6} {'Mean':>6} {'Boroughs <200':>15} {'Boroughs <100':>15}")
print("-" * 60)
for year in sorted(df_all['survey_year'].unique()):
    counts = borough_year_counts.loc[year]
    print(f"{year:<12} {counts.min():>6} {counts.max():>6} {counts.mean():>6.0f} "
          f"{(counts < 200).sum():>15} {(counts < 100).sum():>15}")

# Flag any borough-year combinations below 100 respondents
print("\n\nBOROUGH-YEAR COMBINATIONS BELOW 100 RESPONDENTS (reliability risk):")
print("=" * 80)
found = False
for year in sorted(df_all['survey_year'].unique()):
    counts = borough_year_counts.loc[year]
    low = counts[counts < 100]
    if len(low) > 0:
        found = True
        for borough, count in low.items():
            print(f"  {year} — {borough}: {count} respondents ⚠")
if not found:
    print("  None — all borough-year combinations have 100+ respondents ✓")

RESPONDENTS PER BOROUGH PER YEAR
borough      Barking and Dagenham  Barnet  Bexley  Brent  Bromley  Camden  Croydon  Ealing  Enfield  Greenwich  Hackney  Hammersmith and Fulham  Haringey  Harrow  Havering  Hillingdon  Hounslow  Islington  Kensington and Chelsea  Kingston upon Thames  Lambeth  Lewisham  Merton  Newham  Redbridge  Richmond upon Thames  Southwark  Sutton  Tower Hamlets  Waltham Forest  Wandsworth  Westminster
survey_year                                                                                                                                                                                                                                                                                                                                                                                              
2015-16                      1020     995     511    498      510     514      507     496     1005       1026      508                     516      1027     512       489        

In [30]:
# Confirm the Greenwich Y8 outlier
print("Greenwich respondents by year:")
greenwich = df_all[df_all['borough'] == 'Greenwich'].groupby('survey_year').size()
print(greenwich)

Greenwich respondents by year:
survey_year
2015-16    1026
2016-17     987
2017-18     488
2018-19     486
2019-20     503
2020-21     495
2021-22     496
2022-23     991
dtype: int64


In [31]:
import pyreadstat

# ── Step 5: Cross-year consistency check ─────────────────────────────────────
# Check that category codes mean the same thing across all years
# Focus on key demographic and outcome variables used in modelling

vars_to_check = {
    'Age9':     'Age - nine bands',
    'Gend3':    'Gender',
    'Eth7':     'Ethnicity',
    'Disab3':   'Disability',
    'NSSEC5':   'Socioeconomic',
    'MEMS7GR_ALL': 'Activity level',
    'VolAny':   'Any volunteering',
    'health':   'Self-reported health',
}

# Load metadata for all 8 years
year_meta = {
    '2015-16': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8223-spss 2015-2016\spss\spss28\active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav",   'LA_2021'),
    '2016-17': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8391-spss 2016-2017\spss\spss28\active_lives_survey_nov_16-17_data_year_2_shared_20250106.sav",   'LA_2020'),
    '2017-18': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8651-spss 2017-2018\spss\spss28\active_lives_survey_nov_17-18_data_year_3_shared_20250103.sav",   'LA_Pre2019'),
    '2018-19': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8652-spss 2018-2019\spss\spss28\active_lives_survey_nov_18-19_data_year_4_shared_20250103.sav",   'LA_2019'),
    '2019-20': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8899-spss 2019-2020\spss\spss28\active_lives_survey_nov_19-20_data_year_5_shared_20250103.sav",   'LA_2019'),
    '2020-21': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-8993-spss 2020-2021\spss\spss28\active_lives_survey_nov_20-21_data_year_6_shared_20250103.sav",   'LA_2020'),
    '2021-22': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9136-spss 2021-2022\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav",   'LA_2021'),
    '2022-23': (r"C:\Users\trish\OneDrive - University of Bristol\summer project\Raw Data\UKDA-9288-spss 2022-2023\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",   'LA_2023'),
}

print("Loading metadata for all 8 years...")
all_meta = {}
for year, (path, _) in year_meta.items():
    _, meta = pyreadstat.read_sav(path, metadataonly=True)
    all_meta[year] = meta.variable_value_labels
    print(f"  {year} ✓")

# ── Check each variable ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("CROSS-YEAR CATEGORY CODE CONSISTENCY CHECK")
print("="*70)

for var, description in vars_to_check.items():
    print(f"\n── {var} ({description}) ──")

    # Get labels for each year that has this variable
    year_labels = {}
    for year, labels_dict in all_meta.items():
        if var in labels_dict:
            # Only keep positive codes (exclude sentinels)
            clean = {k: v for k, v in labels_dict[var].items() if k >= 0}
            year_labels[year] = clean
        else:
            year_labels[year] = None

    # Use Y8 as reference
    ref = year_labels.get('2022-23')
    if ref is None:
        print("  Not in Y8 — skipping")
        continue

    print(f"  Y8 reference codes: {ref}")

    # Compare each year to Y8
    all_match = True
    for year, labels in year_labels.items():
        if labels is None:
            print(f"  {year}: ABSENT")
        elif labels == ref:
            print(f"  {year}: ✓ matches Y8")
        else:
            all_match = False
            print(f"  {year}: ⚠ DIFFERENT from Y8")
            # Show what's different
            for code, label in labels.items():
                ref_label = ref.get(code, 'NOT IN Y8')
                if label != ref_label:
                    print(f"    code {code}: '{label}' vs Y8: '{ref_label}'")

Loading metadata for all 8 years...
  2015-16 ✓
  2016-17 ✓
  2017-18 ✓
  2018-19 ✓
  2019-20 ✓
  2020-21 ✓
  2021-22 ✓
  2022-23 ✓

CROSS-YEAR CATEGORY CODE CONSISTENCY CHECK

── Age9 (Age - nine bands) ──
  Y8 reference codes: {1.0: '14-15', 2.0: '16-24', 3.0: '25-34', 4.0: '35-44', 5.0: '45-54', 6.0: '55-64', 7.0: '65-74', 8.0: '75-84', 9.0: '85+'}
  2015-16: ✓ matches Y8
  2016-17: ✓ matches Y8
  2017-18: ✓ matches Y8
  2018-19: ✓ matches Y8
  2019-20: ✓ matches Y8
  2020-21: ✓ matches Y8
  2021-22: ✓ matches Y8
  2022-23: ✓ matches Y8

── Gend3 (Gender) ──
  Y8 reference codes: {1.0: 'Male', 2.0: 'Female', 3.0: 'Other'}
  2015-16: ✓ matches Y8
  2016-17: ✓ matches Y8
  2017-18: ✓ matches Y8
  2018-19: ✓ matches Y8
  2019-20: ✓ matches Y8
  2020-21: ✓ matches Y8
  2021-22: ✓ matches Y8
  2022-23: ✓ matches Y8

── Eth7 (Ethnicity) ──
  Y8 reference codes: {1.0: 'White British', 2.0: 'White Other', 3.0: 'Asian (excl. Chinese)', 4.0: 'Black', 5.0: 'Chinese', 6.0: 'Mixed', 7.0: 'Other 

In [32]:
mask = df_all['survey_year'].isin(['2015-16', '2016-17', '2017-18', '2018-19'])
nssec5_counts = df_all[mask]['NSSEC5'].value_counts().sort_index()
print("NSSEC5 value counts in Y1-Y4:")
print(nssec5_counts)

# Also check if 5.0 exists at all
print(f"\nRows with NSSEC5 = 5.0: {(df_all['NSSEC5'] == 5.0).sum()}")

NSSEC5 value counts in Y1-Y4:
NSSEC5
1.0    39989
2.0    12574
3.0     6459
4.0     6451
5.0     1314
Name: count, dtype: int64

Rows with NSSEC5 = 5.0: 1314


In [33]:
# Recode NSSEC5 = 5.0 → 4.0 for consistency across all years
before = (df_all['NSSEC5'] == 5.0).sum()
df_all['NSSEC5'] = df_all['NSSEC5'].replace(5.0, 4.0)
after = (df_all['NSSEC5'] == 5.0).sum()

print(f"Recoded NSSEC5 = 5.0 → 4.0")
print(f"Before: {before:,} rows with 5.0")
print(f"After:  {after:,} rows with 5.0")

# Verify final value counts
print(f"\nNSSEC5 value counts after recode:")
print(df_all['NSSEC5'].value_counts().sort_index())

Recoded NSSEC5 = 5.0 → 4.0
Before: 1,314 rows with 5.0
After:  0 rows with 5.0

NSSEC5 value counts after recode:
NSSEC5
1.0    77300
2.0    23178
3.0    12397
4.0    13902
Name: count, dtype: int64


In [34]:
# Save cleaned merged dataset
df_all.to_csv('active_lives_london_all_years.csv', index=False)
print(f"✓ Saved: active_lives_london_all_years.csv")
print(f"  Shape: {df_all.shape}")
print(f"  Rows: {len(df_all):,}")
print(f"  Columns: {df_all.shape[1]}")

✓ Saved: active_lives_london_all_years.csv
  Shape: (135497, 610)
  Rows: 135,497
  Columns: 610


In [35]:
import matplotlib.pyplot as plt
import numpy as np

# ── Step 6: Distribution checks on core outcome variables ─────────────────────

# ── 6a: MEMS7GR_ALL — Activity level by year ─────────────────────────────────
print("=" * 60)
print("MEMS7GR_ALL — Activity Level Distribution by Year")
print("=" * 60)
print("Codes: 0=Inactive, 1=Fairly Active, 2=Active\n")

mems_by_year = df_all.groupby('survey_year')['MEMS7GR_ALL'].value_counts(
    normalize=True
).mul(100).round(1).unstack()

mems_by_year.columns = ['Inactive (0)', 'Fairly Active (1)', 'Active (2)']
print(mems_by_year.to_string())

# ── 6b: VolAny — Volunteering rate by year ────────────────────────────────────
print("\n" + "=" * 60)
print("VolAny — Volunteering Rate by Year")
print("=" * 60)
print("Codes: 0=No, 1=Yes\n")

vol_by_year = df_all.groupby('survey_year')['VolAny'].value_counts(
    normalize=True
).mul(100).round(1).unstack()
vol_by_year.columns = ['No (0)', 'Yes (1)']
print(vol_by_year.to_string())

# ── 6c: health — Self-reported health by year ─────────────────────────────────
print("\n" + "=" * 60)
print("health — Self-reported Health by Year")
print("=" * 60)
print("Codes: 1=Very good, 2=Good, 3=Fair, 4=Bad, 5=Very bad\n")

health_by_year = df_all.groupby('survey_year')['health'].value_counts(
    normalize=True
).mul(100).round(1).unstack()
health_by_year.columns = ['Very good(1)', 'Good(2)', 'Fair(3)', 'Bad(4)', 'Very bad(5)']
print(health_by_year.to_string())

# ── 6d: Plausibility check — year on year change ─────────────────────────────
print("\n" + "=" * 60)
print("PLAUSIBILITY CHECK — Year-on-year % point change")
print("=" * 60)

print("\nMEMS7GR_ALL — % Active (code=2):")
active_pct = mems_by_year['Active (2)']
for i in range(1, len(active_pct)):
    change = active_pct.iloc[i] - active_pct.iloc[i-1]
    flag = ' ⚠ large shift' if abs(change) > 5 else ''
    print(f"  {active_pct.index[i-1]} → {active_pct.index[i]}: {change:+.1f}pp{flag}")

print("\nVolAny — % Volunteering (code=1):")
vol_pct = vol_by_year['Yes (1)']
for i in range(1, len(vol_pct)):
    change = vol_pct.iloc[i] - vol_pct.iloc[i-1]
    flag = ' ⚠ large shift' if abs(change) > 5 else ''
    print(f"  {vol_pct.index[i-1]} → {vol_pct.index[i]}: {change:+.1f}pp{flag}")

MEMS7GR_ALL — Activity Level Distribution by Year
Codes: 0=Inactive, 1=Fairly Active, 2=Active

             Inactive (0)  Fairly Active (1)  Active (2)
survey_year                                             
2015-16              21.5               13.0        65.5
2016-17              22.3               13.7        64.1
2017-18              21.3               11.2        67.5
2018-19              21.0               11.2        67.8
2019-20              23.3               10.8        65.9
2020-21              24.0               10.8        65.2
2021-22              22.9               10.2        66.8
2022-23              23.0               10.1        66.9

VolAny — Volunteering Rate by Year
Codes: 0=No, 1=Yes

             No (0)  Yes (1)
survey_year                 
2016-17        78.0     22.0
2017-18        79.5     20.5
2018-19        84.4     15.6
2019-20        82.1     17.9
2020-21        88.0     12.0
2021-22        84.5     15.5
2022-23        82.3     17.7

health — Self-re

In [36]:
# Check VolAny more carefully — compare Y2 Y3 Y4 
# Remember VolAny was renamed from VOLANY in Y2/Y3/Y4
# Check if the drop is consistent across boroughs or concentrated

print("VolAny % by year and inner/outer London:")

# Define inner London boroughs
inner_london = [
    'Camden', 'Greenwich', 'Hackney', 'Hammersmith and Fulham',
    'Islington', 'Kensington and Chelsea', 'Lambeth', 'Lewisham',
    'Newham', 'Southwark', 'Tower Hamlets', 'Wandsworth', 'Westminster',
    'Haringey', 'Waltham Forest'
]

df_all['london_type'] = df_all['borough'].apply(
    lambda x: 'Inner' if x in inner_london else 'Outer'
)

vol_inner_outer = df_all.groupby(['survey_year', 'london_type'])['VolAny'].apply(
    lambda x: f"{x.mean()*100:.1f}%" if x.notna().sum() > 0 else 'N/A'
).unstack()

print(vol_inner_outer.to_string())

# Also check if Y2 and Y3 used different question wording
# by checking value distribution more carefully
print("\nVolAny raw value counts by year:")
for year in ['2016-17', '2017-18', '2018-19', '2019-20']:
    mask = df_all['survey_year'] == year
    counts = df_all[mask]['VolAny'].value_counts(normalize=True).mul(100).round(1)
    print(f"  {year}: {counts.to_dict()}")

VolAny % by year and inner/outer London:
london_type  Inner  Outer
survey_year              
2015-16        N/A    N/A
2016-17      21.4%  22.5%
2017-18      18.7%  22.2%
2018-19      13.7%  17.3%
2019-20      16.1%  19.5%
2020-21      10.9%  13.1%
2021-22      14.6%  16.5%
2022-23      16.3%  19.0%

VolAny raw value counts by year:


C:\Users\trish\AppData\Local\Temp\ipykernel_39472\422751528.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_all['london_type'] = df_all['borough'].apply(


  2016-17: {0.0: 78.0, 1.0: 22.0}
  2017-18: {0.0: 79.5, 1.0: 20.5}
  2018-19: {0.0: 84.4, 1.0: 15.6}
  2019-20: {0.0: 82.1, 1.0: 17.9}
